# BRIDGE Dataset Preparation and Analysis

This notebook prepares the TutorState dataset and keeps exploration, preprocessing, export, and analysis in separate segments. Run it from the project directory, from top to bottom. Use **Shift+Enter** to run a cell.

## Notebook Guide

| Segment | Purpose |
|---|---|
| [1. Setup and Raw Exploration](#segment-1) | Load the raw dataset and inspect its fields |
| [2. Error Label Cleaning](#segment-2) | Standardise `e`, inspect exclusions, and remove the approved training duplicate |
| [3. Teaching Strategy Mapping](#segment-3) | Map `z_what` and retain records with both labels mapped |
| [4. Lesson Topic Standardisation](#segment-4) | Clean topic names and apply approved case assignments and spelling corrections |
| [5. Topic Merge Preview](#segment-5) | Review the nine proposed topic merges without exporting them |
| [6. Final Schema and Export](#segment-6) | Reformat histories and produce the shared 12-field JSONL files |
| [7. Conversation Context Analysis](#segment-7) | Screen question visibility and response context without changing data |

### Dataset Stages

| Variable / file | Records | Meaning |
|---|---:|---|
| `dataset` | 700 | Original BRIDGE data |
| `cleaned_dataset` | 669 | Intermediate records after error-label cleaning and training-duplicate removal |
| `final_dataset` | 660 | Label- and topic-cleaned records, retaining original histories for audit |
| `topic_merged_preview_dataset` | 660 | Optional 115-topic preview; not used by the export segment |
| `simplified_dataset` | 660 | Final 12-field records with reformatted `c_h` |
| `data/bridge_excluded.jsonl` | 40 | Combined exclusion audit |

The shared split files contain **397 train, 68 validation, and 195 test records**. Earlier export cells produce intermediate files; **Segment 6 writes the final 12-field exports**. The nine proposed topic merges remain preview-only, so exported data currently has **124 unique cleaned topics**.

<a id="segment-1"></a>

## Segment 1: Setup and Raw Exploration

### 1.1 Load the Raw Dataset

Run the setup and loading cells first. The remaining cells in this segment inspect raw values without changing the dataset.


In [ ]:
# Run this cell once if the notebook kernel does not have datasets installed.
# %pip install datasets

In [ ]:
import os
from pathlib import Path
from pprint import pprint

# Reuse the dataset cache downloaded for this project.
os.environ.setdefault("HF_HOME", str(Path.cwd() / ".cache" / "huggingface"))
from datasets import load_dataset

dataset = load_dataset("rose-e-wang/bridge")
print(dataset)

### 1.2 Count Raw Error Annotations

Run this section after loading the dataset. Counts include train, validation, and test; exact text values are kept separate.


In [ ]:
all_e = [
    value
    for split in ("train", "validation", "test")
    for value in dataset[split]["e"]
]
unique_e = sorted(set(all_e))
print("Number of unique values:", len(unique_e))

In [1]:
from collections import Counter

# Count exact e values across train, validation, and test.
e_counts = Counter(
    value
    for split in dataset.values()
    for value in split["e"]
)

print("e | counts")
print("-" * 60)
for value, count in e_counts.most_common():
    label = repr(value)[1:-1]
    print(f"{label} | {count}")


e | counts
------------------------------------------------------------
guess | 239
right-idea | 129
diagnose | 128
careless | 55
imprecise | 48
misinterpret | 45
The student did not make a mistake. | 5
End session | 4
It doesn't seem like the student made a mistake. | 3
no error | 2
The student did not make an error. | 2
Student unresponsive | 1
The student did not make a mistake. Rather, he was telling the tutor that he now understands. | 1
student stopped responding | 1
Student did not make a mistake, but also does not seem to be answering | 1
student is not wrong | 1
Ending session | 1
Student answered question | 1
Student did not make a mistake. | 1
Student did not make a mistake | 1
student is ending the session | 1
Student ready to end session. | 1
The student did not make a mistake. The student just said that he wasn't at the computer. | 1
Student is correct\n | 1
No mistake, student is just ending the session. | 1
End session. | 1
Correct Answer | 1
It seems like the student i

### 1.3 Select an Example Record

Available splits: `train`, `validation`, and `test`. Row indices start at 0. Rerun this cell after changing the selection, then rerun any attribute cells you want to inspect.


In [ ]:
split_name = "train"
row_index = 0

example = dataset[split_name][row_index]
print(f"Selected: {split_name}, row {row_index}")
print("Attributes:", list(example.keys()))
print("Column types:")
pprint(dataset[split_name].features)

#### 1.3.1 Case Identifier (`c_id`)


In [ ]:
pprint(example["c_id"], width=100, sort_dicts=False)

#### 1.3.2 Conversation History (`c_h`)


In [ ]:
pprint(example["c_h"], width=100, sort_dicts=False)

#### 1.3.3 Revised Continuation (`c_revision`)


In [ ]:
pprint(example["c_revision"], width=100, sort_dicts=False)

#### 1.3.4 Error Annotation (`e`)


In [ ]:
pprint(example["e"], width=100, sort_dicts=False)

#### 1.3.5 Lesson Topic (`lesson_topic`)


In [ ]:
pprint(example["lesson_topic"], width=100, sort_dicts=False)

#### 1.3.6 Teaching Strategy (`z_what`)


In [ ]:
pprint(example["z_what"], width=100, sort_dicts=False)

#### 1.3.7 Strategy Intention (`z_why`)


In [ ]:
pprint(example["z_why"], width=100, sort_dicts=False)

#### 1.3.8 Original Tutor Response (`c_r`)


In [ ]:
pprint(example["c_r"], width=100, sort_dicts=False)

#### 1.3.9 Expert Revised Response (`c_r_`)


In [ ]:
pprint(example["c_r_"], width=100, sort_dicts=False)

### 1.4 Inspect an Attribute Across Multiple Rows

Change `attribute`, `split_name`, or `number_of_rows` below. Use `len(dataset[split_name])` as the number of rows to print the whole split.


In [ ]:
attribute = "e"
number_of_rows = 5

for index, value in enumerate(dataset[split_name][attribute]):
    if index >= number_of_rows:
        break
    print(f"Row {index}:")
    pprint(value, width=100, sort_dicts=False)
    print()

<a id="segment-2"></a>

## Segment 2: Error Label Cleaning

### 2.1 Define the Cleaning Rules

This segment reuses `dataset` and produces the 669-record error-cleaned intermediate dataset. The shared dataset is further filtered in Segment 3 and simplified in Segment 6.

Keep the six standard labels: `guess`, `careless`, `right-idea`, `misinterpret`, `diagnose`, and `imprecise`. Map the explicitly approved correct/no-mistake annotations to `no_error`.

The `no_error_annotations` list includes **both effort-related annotations**, including the approved wording that omits **that** after **indicating**. No special follow-up mapping is needed. These are annotation-based labels, not independently verified mathematical judgments.

Matching ignores case and whitespace differences; it never uses fuzzy matching or keyword classification. Non-diagnostic, ambiguous, tutor-error, and unexpected annotations are excluded. An additional approved rule removes the training copy of record `2964114_96` while retaining its identical test copy.

**Evaluation only:** `e_clean` is a target label. Do not pass it as an input to the Student Analyser during inference.


In [16]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import json
import pandas as pd
from datasets import DatasetDict, Features, Value
from IPython.display import display

if "dataset" not in globals():
    raise RuntimeError("Run the existing dataset-loading cell before this section.")

split_names = ("train", "validation", "test")
assert set(dataset) == set(split_names), "Unexpected split names."

# Keep a reference and an independent snapshot to verify raw data preservation.
original_dataset_reference = dataset
original_rows_snapshot = {
    split: deepcopy(list(dataset[split])) for split in split_names
}
original_features_snapshot = {
    split: deepcopy(dataset[split].features) for split in split_names
}

recurring_labels = (
    "guess", "careless", "right-idea", "misinterpret", "diagnose", "imprecise"
)
approved_labels = (*recurring_labels, "no_error")

def normalise_error_text(value):
    """Normalise matching text only; leave original annotations untouched."""
    if not isinstance(value, str):
        return None
    return " ".join(value.split()).casefold()

In [17]:
# Approved exact annotations, including both effort-related wordings.
no_error_annotations = ['The student did not make a mistake.',
 "It doesn't seem like the student made a mistake.",
 'no error',
 'The student did not make an error.',
 'The student did not make a mistake. Rather, he was telling the tutor that he now understands.',
 'student is not wrong',
 'Student did not make a mistake.',
 'Student did not make a mistake',
 'Student is correct',
 'Correct Answer',
 'The student did not make a mistake in this given segment.',
 'Student did not make a mistake or have a misconception.',
 'The student did not make a mistake and the tutor did not indicate that the student had made a '
 'mistake. Maybe the word "effort" was mistakenly flagged as the tutor indicating that the student '
 'had made a mistake.',
 'No mistake!',
 "I don't see that the student made an error",
 'The student did not make an error',
 "The student did not make a mistake, and I don't think the tutor indicated that the student made "
 'a mistake. I think the "Great effort" was mistakenly flagged as the tutor indicating that the '
 'student made a mistake.',
 'Student understands',
 'The student did not make a mistake and the tutor did not indicate that the student had made a '
 'mistake. Maybe the word "effort" was mistakenly flagged as the tutor indicating  the student had '
 'made a mistake.']

# Exclusion reasons for annotations outside the seven evaluation classes.
exclusion_annotations = {'non_diagnostic': ['End session',
                    'Student unresponsive',
                    'student stopped responding',
                    'Student did not make a mistake, but also does not seem to be answering',
                    'Ending session',
                    'student is ending the session',
                    'Student ready to end session.',
                    "The student did not make a mistake. The student just said that he wasn't at "
                    'the computer.',
                    'No mistake, student is just ending the session.',
                    'End session.',
                    'It seems like the student is reluctant to answer and wants to ends the '
                    'session. Does not seem like the student made mistake though.',
                    'student stops responding',
                    'Student ending session',
                    "tutor didn't ask for answer",
                    'Student needs to go',
                    'The student did not make a mistake. The student was telling the tutor that '
                    'there was nothing more the tutor could help with today.'],
 'ambiguous': ['Student answered question',
               'Answered in another language',
               "The student didn't make a mistake. I don't know whether the tutor is acknowledging "
               "the student's response as a mistake because sometimes the tutor would give +3 "
               "points for a correct answer and +1 for an incorrect answer. I'm not sure if that's "
               'for all tutors or only some tutors.',
               'unrelated to content, but student answered question',
               'Student did not make a mistake. Possible that the student did make a mistake '
               "earlier and the tutor was explaining why the student's answer was incorrect.",
               "I think the student did get the correct answer. I don't know for sure based on the "
               'context.'],
 'tutor_error': ['The student has not made a mistake yet - in fact, I am pretty sure it was the '
                 'tutor who made the mistake. The tutor said "the exponent would be 10^4", when '
                 "I'm pretty sure the exponent is just 4. Maybe that's why the student only "
                 '"kinda" gets it.',
                 'The tutor is incorrect and the student has the correct response.',
                 'The tutor made a slight mistake in his explanation. He says "We need to subtract '
                 '24 from 15" which means 15 - 24 but then asks for 24 - 15. This could have '
                 'confused the student.']}

In [18]:
# Convert annotation lists into lookup tables once.
retained_label_mapping = {
    normalise_error_text(label): label for label in recurring_labels
}
retained_label_mapping.update({
    normalise_error_text(annotation): "no_error"
    for annotation in no_error_annotations
})
exclusion_mapping = {}
for reason, annotations in exclusion_annotations.items():
    for annotation in annotations:
        key = normalise_error_text(annotation)
        assert key not in exclusion_mapping, "Conflicting exclusion rules."
        exclusion_mapping[key] = reason
assert not (retained_label_mapping.keys() & exclusion_mapping.keys())

def clean_error_label(original_value):
    """Return audit fields using exact matching after text normalisation."""
    key = normalise_error_text(original_value)
    cleaned_label = retained_label_mapping.get(key)
    included = cleaned_label is not None
    return {
        "e_original": original_value,
        "e_clean": cleaned_label,
        "exclusion_reason": None if included else exclusion_mapping.get(key, "unmapped"),
        "include_for_evaluation": included,
    }

### 2.2 Apply Cleaning and Remove the Approved Duplicate

First verify that the identified train and test records are identical across all original fields. Only the training copy receives `exclusion_reason = "duplicate_train_test"`.

Apply the label and duplicate rules together, then partition the records once into `cleaned_dataset` and `excluded_dataset`. `annotated_dataset` keeps all decisions. Every dataset preserves the original split names and fields, including the unchanged `e`. Added fields record the cleaned label, original annotation, exclusion decision, and source row.


In [19]:
duplicate_record_id = "2964114_96"
train_duplicate_rows = [
    index for index, row in enumerate(dataset["train"])
    if row["c_id"] == duplicate_record_id
]
test_duplicate_rows = [
    index for index, row in enumerate(dataset["test"])
    if row["c_id"] == duplicate_record_id
]
assert len(train_duplicate_rows) == len(test_duplicate_rows) == 1
train_duplicate_index = train_duplicate_rows[0]
test_duplicate_index = test_duplicate_rows[0]
assert dataset["train"][train_duplicate_index] == dataset["test"][test_duplicate_index]
duplicate_label = clean_error_label(dataset["train"][train_duplicate_index]["e"])["e_clean"]
assert duplicate_label in approved_labels

# Split and source row index identify records; c_id alone is not always unique.
def annotate_record(record, row_index, original_split):
    decision = clean_error_label(record["e"])
    if (original_split, row_index) == ("train", train_duplicate_index):
        decision.update(
            e_clean=None,
            exclusion_reason="duplicate_train_test",
            include_for_evaluation=False,
        )
    return {
        **decision,
        "original_split": original_split,
        "source_row_index": row_index,
    }

In [20]:
added_features = {
    "e_original": Value("string"),
    "e_clean": Value("string"),
    "exclusion_reason": Value("string"),
    "include_for_evaluation": Value("bool"),
    "original_split": Value("string"),
    "source_row_index": Value("int64"),
}
annotated_dataset = DatasetDict()
for split in split_names:
    assert not (set(added_features) & set(dataset[split].column_names))
    annotated_dataset[split] = dataset[split].map(
        annotate_record,
        with_indices=True,
        fn_kwargs={"original_split": split},
        features=Features({**dataset[split].features, **added_features}),
        keep_in_memory=True,
        load_from_cache_file=False,
        desc=f"Clean and audit {split}",
    )

cleaned_dataset = DatasetDict({
    split: annotated_dataset[split].filter(
        lambda row: row["include_for_evaluation"],
        keep_in_memory=True, load_from_cache_file=False,
        desc=f"Retain approved records: {split}",
    ) for split in split_names
})
excluded_dataset = DatasetDict({
    split: annotated_dataset[split].filter(
        lambda row: not row["include_for_evaluation"],
        keep_in_memory=True, load_from_cache_file=False,
        desc=f"Collect exclusions: {split}",
    ) for split in split_names
})

### 2.3 Inspect Error Cleaning Results

All tables reflect the final mapping and training-duplicate exclusion. The excluded-record audit retains the original annotation and split, including any unexpected values. Examples show original annotations alongside their cleaned labels.


In [21]:
split_summary = pd.DataFrame([
    {
        "split": split,
        "original": len(dataset[split]),
        "retained": len(cleaned_dataset[split]),
        "excluded": len(excluded_dataset[split]),
    }
    for split in split_names
]).set_index("split")
display(split_summary)

counts_by_split = {
    split: Counter(cleaned_dataset[split]["e_clean"])
    for split in split_names
}
label_counts_by_split = pd.DataFrame({
    split: [counts_by_split[split][label] for label in approved_labels]
    for split in split_names
}, index=pd.Index(approved_labels, name="e_clean"))
label_distribution = label_counts_by_split.sum(axis=1).rename("counts").to_frame()
print("Seven-class distribution across all splits:")
display(label_distribution)
print("Seven-class distribution within each split:")
display(label_counts_by_split)

Seven-class distribution across all splits:
Seven-class distribution within each split:


,original,retained,excluded
split,,,
train,419,402,17
validation,71,69,2
test,210,198,12


,counts
e_clean,
guess,238
careless,55
right-idea,129
misinterpret,45
diagnose,128
imprecise,48
no_error,26


,train,validation,test
e_clean,,,
guess,151,25,62
careless,29,3,23
right-idea,78,15,36
misinterpret,26,4,15
diagnose,76,15,37
imprecise,26,5,17
no_error,16,2,8


In [22]:
audit_columns = [
    "original_split", "source_row_index", "c_id", "e_original", "e_clean",
    "exclusion_reason", "include_for_evaluation",
]
retained_audit = pd.DataFrame([
    {column: row[column] for column in audit_columns}
    for split in split_names for row in cleaned_dataset[split]
], columns=audit_columns)
excluded_audit = pd.DataFrame([
    {column: row[column] for column in audit_columns}
    for split in split_names for row in excluded_dataset[split]
], columns=audit_columns)

exclusion_reasons = ("non_diagnostic", "ambiguous", "tutor_error", "unmapped", "duplicate_train_test")
exclusion_reason_counts = (
    excluded_audit["exclusion_reason"].value_counts()
    .reindex(exclusion_reasons, fill_value=0)
    .rename_axis("exclusion_reason").rename("counts").to_frame()
)
print("Exclusion reasons across all splits:")
display(exclusion_reason_counts)

excluded_annotation_summary = (
    excluded_audit.groupby(["exclusion_reason", "e_original"], dropna=False)
    .size().rename("counts").reset_index()
    .sort_values(["exclusion_reason", "counts"], ascending=[True, False])
)
print("Excluded raw annotations and their frequencies:")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(excluded_annotation_summary)

unmapped_annotations = (
    excluded_audit.loc[excluded_audit["exclusion_reason"].eq("unmapped"), "e_original"]
    .value_counts(dropna=False).rename_axis("unexpected_e")
    .rename("counts").reset_index()
)
if unmapped_annotations.empty:
    print("No unexpected unmapped annotations.")
else:
    print("WARNING: Unexpected annotations were excluded without guessing a class.")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(unmapped_annotations)

print("Examples of original and cleaned labels (up to two per class):")
with pd.option_context("display.max_colwidth", None):
    display(retained_audit.groupby("e_clean", sort=False).head(2)[
        ["original_split", "c_id", "e_original", "e_clean"]
    ])

Exclusion reasons across all splits:
Excluded raw annotations and their frequencies:
No unexpected unmapped annotations.
Examples of original and cleaned labels (up to two per class):


,counts
exclusion_reason,
non_diagnostic,20
ambiguous,7
tutor_error,3
unmapped,0
duplicate_train_test,1


,exclusion_reason,e_original,counts
0,ambiguous,Answered in another language,1
1,ambiguous,I think the student did get the correct answer. I don't know for sure based on the context.,1
2,ambiguous,Student answered question,1
3,ambiguous,Student answered question,1
4,ambiguous,Student did not make a mistake. Possible that the student did make a mistake earlier and the tutor was explaining why the student's answer was incorrect.,1
5,ambiguous,The student didn't make a mistake. I don't know whether the tutor is acknowledging the student's response as a mistake because sometimes the tutor would give +3 points for a correct answer and +1 for an incorrect answer. I'm not sure if that's for all tutors or only some tutors.,1
6,ambiguous,"unrelated to content, but student answered question",1
7,duplicate_train_test,guess,1
8,non_diagnostic,End session,4
9,non_diagnostic,End session\n,1


,original_split,c_id,e_original,e_clean
0,train,4110653_19,guess,guess
1,train,2950057_118,diagnose,diagnose
2,train,2989485_22,guess,guess
3,train,2928616_65,careless,careless
6,train,2916421_82,right-idea,right-idea
8,train,6172033_30,misinterpret,misinterpret
10,train,4110653_56,right-idea,right-idea
13,train,4015974_13,imprecise,imprecise
18,train,2950026_44,careless,careless
20,train,4293213_83,diagnose,diagnose


### 2.4 Validate Error Cleaning

Check all 700 raw rows, unchanged original fields, original split membership, disjoint partitions, all seven labels, and absence of identical full records in the cleaned data. The excluded training duplicate remains in the audit, while its test copy stays in the evaluation data.

Compare the six original recurring-label frequencies before duplicate removal, then show the effect of the single training exclusion. Counts are calculated from the records; the earlier approximate 670/30 expectation is only a comparison.


In [23]:
validation_passed = False
expected_original_split_sizes = {"train": 419, "validation": 71, "test": 210}
original_count = sum(len(dataset[split]) for split in split_names)
retained_count = sum(len(cleaned_dataset[split]) for split in split_names)
excluded_count = sum(len(excluded_dataset[split]) for split in split_names)
assert dataset is original_dataset_reference
assert sum(len(dataset[split]) for split in split_names) == 700
assert retained_count + excluded_count == 700
assert set(cleaned_dataset) == set(excluded_dataset) == set(split_names)
assert set(retained_audit["e_clean"]) == set(approved_labels)
assert retained_audit["exclusion_reason"].isna().all()
assert retained_audit["include_for_evaluation"].eq(True).all()
assert excluded_audit["e_clean"].isna().all()
assert excluded_audit["include_for_evaluation"].eq(False).all()
assert excluded_audit["exclusion_reason"].isin(exclusion_reasons).all()

seen_keys = set()
retained_signatures = []
for split in split_names:
    assert list(dataset[split]) == original_rows_snapshot[split]
    assert dataset[split].features == original_features_snapshot[split]
    assert len(dataset[split]) == expected_original_split_sizes[split]
    assert len(annotated_dataset[split]) == len(dataset[split])
    for partition_name, partition in (("retained", cleaned_dataset), ("excluded", excluded_dataset)):
        for column, feature in original_features_snapshot[split].items():
            assert partition[split].features[column] == feature
        for row in partition[split]:
            assert row["original_split"] == split
            key = (split, row["source_row_index"])
            assert key not in seen_keys, "A source row was duplicated across partitions."
            seen_keys.add(key)
            original = original_rows_snapshot[split][row["source_row_index"]]
            raw_fields = {column: row[column] for column in original}
            assert raw_fields == original
            assert row == annotated_dataset[split][row["source_row_index"]]
            assert row["e_original"] == row["e"] == original["e"]
            expected = clean_error_label(original["e"])
            if key == ("train", train_duplicate_index):
                expected.update(e_clean=None, exclusion_reason="duplicate_train_test",
                                include_for_evaluation=False)
            assert {column: row[column] for column in expected} == expected
            assert row["include_for_evaluation"] == (partition_name == "retained")
            if partition_name == "retained":
                retained_signatures.append(json.dumps(raw_fields, sort_keys=True, ensure_ascii=False))
assert seen_keys == {
    (split, index) for split in split_names for index in range(len(dataset[split]))
}
assert len(retained_signatures) == len(set(retained_signatures)), "Duplicate content remains in cleaned data."
assert duplicate_record_id not in set(cleaned_dataset["train"]["c_id"])
assert list(cleaned_dataset["test"]["c_id"]).count(duplicate_record_id) == 1
assert excluded_audit["exclusion_reason"].eq("duplicate_train_test").sum() == 1
assert len(cleaned_dataset["test"]) == sum(
    clean_error_label(row["e"])["include_for_evaluation"] for row in dataset["test"]
)

label_only_distribution = Counter(
    clean_error_label(row["e"])["e_clean"]
    for split in split_names for row in dataset[split]
    if clean_error_label(row["e"])["include_for_evaluation"]
)
expected_after_deduplication = label_only_distribution.copy()
expected_after_deduplication[duplicate_label] -= 1
assert label_distribution["counts"].to_dict() == dict(expected_after_deduplication)
assert sum(label_distribution["counts"]) == retained_count
assert sum(exclusion_reason_counts["counts"]) == excluded_count
validation_passed = True
print(f"Final validation passed: {retained_count} retained + {excluded_count} excluded = 700.")
print("The cleaned train/validation/test data contains no identical full records.")
print("Raw train/test duplicate is preserved only in the raw dataset and exclusion audit.")
comparison = pd.DataFrame({
    "before_duplicate_removal": pd.Series(label_only_distribution),
    "final_count": label_distribution["counts"],
}).reindex(approved_labels)
display(comparison)

Final validation passed: 669 retained + 31 excluded = 700.
The cleaned train/validation/test data contains no identical full records.
Raw train/test duplicate is preserved only in the raw dataset and exclusion audit.


,before_duplicate_removal,final_count
guess,239,238
careless,55,55
right-idea,129,129
misinterpret,45,45
diagnose,128,128
imprecise,48,48
no_error,26,26


In [24]:
expected_recurring_counts = {
    "guess": 239, "right-idea": 129, "diagnose": 128,
    "careless": 55, "imprecise": 48, "misinterpret": 45,
}
recurring_count_comparison = pd.DataFrame([
    {
        "e_clean": label,
        "earlier_count": expected,
        "before_duplicate_removal": label_only_distribution[label],
        "final_count": int(label_distribution.loc[label, "counts"]),
    }
    for label, expected in expected_recurring_counts.items()
]).set_index("e_clean")
display(recurring_count_comparison)
if any(label_only_distribution[label] != expected
       for label, expected in expected_recurring_counts.items()):
    print("WARNING: Label-only counts differ from the earlier exploration.")
else:
    print("Original recurring-label counts match. Removing the train duplicate reduces guess by one.")

approximate_expectation = pd.DataFrame({
    "original_approximate_expectation": [670, 30],
    "final_actual": [retained_count, excluded_count],
}, index=pd.Index(["retained", "excluded"], name="partition"))
approximate_expectation["difference"] = (
    approximate_expectation["final_actual"]
    - approximate_expectation["original_approximate_expectation"]
)
display(approximate_expectation)
print("The approved training-duplicate exclusion moves one additional record to exclusions.")

Original recurring-label counts match. Removing the train duplicate reduces guess by one.
The approved training-duplicate exclusion moves one additional record to exclusions.


,earlier_count,before_duplicate_removal,final_count
e_clean,,,
guess,239,239,238
right-idea,129,129,129
diagnose,128,128,128
careless,55,55,55
imprecise,48,48,48
misinterpret,45,45,45


,original_approximate_expectation,final_actual,difference
partition,,,
retained,670,669,-1
excluded,30,31,1


### 2.5 Export the Error-Cleaned Intermediate Data

This export records the error-cleaning stage: 669 retained records and 31 exclusions. It preserves raw fields and uses UTF-8. Each JSONL file is reloaded for verification.

These are intermediate exports. Segments 3, 4, and 6 update the processed files; the final shared schema is produced in Segment 6. Run the full pipeline in order when regenerating the shared files.


In [25]:
assert validation_passed, "Complete the validation section before exporting."
output_directory = Path("data")
output_directory.mkdir(parents=True, exist_ok=True)

def write_jsonl(path, records):
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")

export_manifest = []
for split in split_names:
    output_path = output_directory / f"bridge_{split}.jsonl"
    write_jsonl(output_path, cleaned_dataset[split])
    # Round-trip verification also checks nested original columns and Unicode.
    with output_path.open(encoding="utf-8") as handle:
        saved_records = [json.loads(line) for line in handle]
    assert saved_records == list(cleaned_dataset[split])
    export_manifest.append({"file": output_path.as_posix(), "records": len(saved_records)})

excluded_path = output_directory / "bridge_excluded.jsonl"
all_excluded_records = [row for split in split_names for row in excluded_dataset[split]]
write_jsonl(excluded_path, all_excluded_records)
with excluded_path.open(encoding="utf-8") as handle:
    saved_exclusions = [json.loads(line) for line in handle]
assert saved_exclusions == all_excluded_records
export_manifest.append({"file": excluded_path.as_posix(), "records": len(saved_exclusions)})
print("Saved and verified processed JSONL exports:")
display(pd.DataFrame(export_manifest))

Saved and verified processed JSONL exports:


,file,records
0,data/bridge_train.jsonl,402
1,data/bridge_validation.jsonl,69
2,data/bridge_test.jsonl,198
3,data/bridge_excluded.jsonl,31


### 2.6 Inspect Strategy Annotations After Error Cleaning

Count exact `z_what` annotations in the 669-record `cleaned_dataset`, before strategy-specific exclusions. The following history inspection shows the selected resource, visual-aid, and calculator cases. Original annotations and histories are preserved.


In [1]:
from collections import Counter

z_what_counts = Counter(
    value
    for split in cleaned_dataset.values()
    for value in split["z_what"]
)
unique_z_what = sorted(z_what_counts)

assert sum(z_what_counts.values()) == sum(len(split) for split in cleaned_dataset.values())
print("Retained records:", sum(z_what_counts.values()))
print("Number of unique z_what values:", len(unique_z_what))
print("z_what | counts")
print("-" * 60)
for value in unique_z_what:
    print(f"{value} | {z_what_counts[value]}")


Retained records: 669
Number of unique z_what values: 21
z_what | counts
------------------------------------------------------------
Allowed the use of a calculator | 1
Asked the student to fill out the feedback form. | 1
Explained the online interface | 1
Provided a resource | 1
Provided a visual aid | 8
Provided a visual aid. | 6
Provided a visual aid. The student should then be able to see where the thousands place is. | 1
Reminded the student to fill out the feedback form. | 1
Said goodbye to the student | 5
Told the student the answer. I don't think it's too important to derive how there are 24 hours in a day. It seems like something you just need to know. | 1
Told the student the value of pi rounded to 2 decimal places. The student needs to know this if he is to round pi, which I think may be the followup question | 1
affirm_correct_answer | 28
ask_question | 204
encourage_student | 22
explain_concept | 136
provide_correction | 40
provide_example | 20
provide_hint | 63
provide_s

In [1]:
import pandas as pd
from IPython.display import display, Markdown

# Exact matches in the final cleaned records; punctuation variants stay separate.
selected_z_what = [
    "Provided a resource",
    "Provided a visual aid",
    "Provided a visual aid.",
    "Provided a visual aid. The student should then be able to see where the thousands place is.",
    "Allowed the use of a calculator",
]

selected_history_records = {
    annotation: [
        (split_name, record)
        for split_name, split in cleaned_dataset.items()
        for record in split
        if record["z_what"] == annotation
    ]
    for annotation in selected_z_what
}

for annotation, records in selected_history_records.items():
    display(Markdown(f"#### {annotation} ({len(records)} records)"))
    for number, (split_name, record) in enumerate(records, start=1):
        print(f"Example {number} | split: {split_name} | c_id: {record['c_id']} "
              f"| source row: {record['source_row_index']} | e_clean: {record['e_clean']}")
        # Show every message and every c_h field, with full text and no truncation.
        with pd.option_context(
            "display.max_rows", None,
            "display.max_columns", None,
            "display.max_colwidth", None,
        ):
            display(pd.DataFrame(record["c_h"]))


#### Provided a resource (1 records)

Example 1 | split: train | c_id: 6383453_36 | source row: 232 | e_clean: guess


,id,text,time,user
0,36,Consider the first digit in the given number.,09:51,tutor
1,37,How many times 4 go in 6?,10:03,tutor
2,38,Have you learned multiplication tables?,10:25,tutor
3,39,6,10:37,student


#### Provided a visual aid (8 records)

Example 1 | split: train | c_id: 2979589_63 | source row: 11 | e_clean: guess


...

Example 1 | split: validation | c_id: 4169950_31 | source row: 21 | e_clean: guess


,id,text,time,user
0,31,Here we go through the next question.,19:48,tutor
1,32,ok,19:58,student
2,33,What digit is in the thousands place?,20:18,tutor
3,34,8?,20:50,student


#### Allowed the use of a calculator (1 records)

Example 1 | split: test | c_id: 2949298_23 | source row: 208 | e_clean: guess


,id,text,time,user
0,23,245,08:56,student
1,24,Nice try! But your answer is incorrect!,09:34,tutor
2,25,What is the sum of 27 and 16?,10:18,tutor
3,26,11664.,10:45,student


<a id="segment-3"></a>

## Segment 3: Teaching Strategy Mapping

### 3.1 Define the Action Mappings

Reuse the intermediate 669-record `cleaned_dataset` from the error-label cleaning above. The raw `dataset`, `cleaned_dataset`, existing fields, and their split assignments are preserved. These cells do not reload BRIDGE or repeat error-label preprocessing.

| Action | Meaning |
|---|---|
| `PROBE` | Ask questions about the student's reasoning or understanding |
| `HINT` | Give a clue without directly revealing the answer |
| `EXPLAIN` | Explain a concept, method, or example |
| `SIMPLIFY` | Break down a task or offer a more manageable similar problem |
| `CORRECT` | Directly correct the student's mistake |
| `AFFIRM` | Confirm a correct response |
| `ENCOURAGE` | Motivate or reassure the student |

The ten proposal strategies define the strict audit subset. Strict and extended datasets are intermediate analysis variables; the shared exports are the three split files and one exclusion compilation. Visual aids and resource provision receive **provisional HINT** mappings and are included only in the exploratory extended subset. These 16 provisional examples remain flagged for manual review through `mapping_status = "provisional"`. The two factual-assistance annotations are explicitly approved as `CORRECT`, with `mapping_status = "manual_approved"` and `mapping_source = "manual_user_rule"`. They are included in extended evaluation, while the strict benchmark remains limited to the ten proposal strategies. Calculator assistance and non-pedagogical interactions are excluded from planner evaluation, while diagnosis eligibility is retained.

Normalisation changes matching text only: collapse whitespace, strip its edges, and ignore case. Punctuation variants are listed explicitly; no fuzzy or keyword classification is used.

**Evaluation:** labels measure agreement with one expert-selected action, not the only valid teaching action. The planner's inference inputs are conversation history, analyser output, and updated student-state memory. Never supply `z_what`, `z_what_clean`, `z_why`, expert tutor responses, or future turns as planner inputs.


In [1]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import json
import pandas as pd
from datasets import DatasetDict, Features, Value
from IPython.display import display, Markdown

if "cleaned_dataset" not in globals():
    raise RuntimeError("Run the existing error-label preprocessing cells first.")
planner_split_names = ("train", "validation", "test")
assert set(cleaned_dataset) == set(planner_split_names)
planner_source_reference = cleaned_dataset
planner_source_rows = {
    split: deepcopy(list(cleaned_dataset[split])) for split in planner_split_names
}
planner_source_features = {
    split: deepcopy(cleaned_dataset[split].features) for split in planner_split_names
}
# Preserve the raw dataset object too when this notebook has loaded it.
planner_raw_reference = globals().get("dataset")
planner_actions = ("PROBE", "HINT", "EXPLAIN", "SIMPLIFY", "CORRECT", "AFFIRM", "ENCOURAGE")
planner_mapping_statuses = ("standard", "manual_approved", "provisional", "excluded", "unmapped")

def normalise_strategy_text(value):
    """Normalise matching keys without changing the original annotation."""
    return " ".join(value.split()).casefold() if isinstance(value, str) else None

standard_strategy_mapping = {
    "ask_question": "PROBE",
    "provide_hint": "HINT",
    "explain_concept": "EXPLAIN",
    "provide_strategy": "EXPLAIN",
    "provide_example": "EXPLAIN",
    "simplify_question": "SIMPLIFY",
    "provide_similar_problem": "SIMPLIFY",
    "provide_correction": "CORRECT",
    "affirm_correct_answer": "AFFIRM",
    "encourage_student": "ENCOURAGE",
}
provisional_strategy_mapping = {
    "Provided a visual aid": "HINT",
    "Provided a visual aid.": "HINT",
    "Provided a visual aid. The student should then be able to see where the thousands place is.": "HINT",
    "Provided a resource": "HINT",
}
excluded_strategy_mapping = {
    "Allowed the use of a calculator": "tool_assistance",
    "Asked the student to fill out the feedback form.": "non_pedagogical",
    "Reminded the student to fill out the feedback form.": "non_pedagogical",
    "Explained the online interface": "non_pedagogical",
    "Said goodbye to the student": "non_pedagogical",
}
# Exact annotation mappings explicitly approved after inspecting both conversations.
approved_manual_strategy_mapping = {
    "Told the student the answer. I don't think it's too important to derive how there are 24 hours in a day. It seems like something you just need to know.": "CORRECT",
    'Told the student the value of pi rounded to 2 decimal places. The student needs to know this if he is to round pi, which I think may be the followup question': "CORRECT",
}


In [2]:
# Build one lookup with explicit outcomes and provenance.
strategy_rules = {}

def register_strategy(annotation, action, status, source, reason=None):
    key = normalise_strategy_text(annotation)
    assert key not in strategy_rules, f"Conflicting mapping for {annotation!r}"
    strategy_rules[key] = {
        "z_what_clean": action,
        "mapping_status": status,
        "mapping_source": source,
        "z_what_exclusion_reason": reason,
    }

for annotation, action in standard_strategy_mapping.items():
    register_strategy(annotation, action, "standard", "proposal_mapping")
for annotation, action in provisional_strategy_mapping.items():
    register_strategy(annotation, action, "provisional", "manual_project_rule")
for annotation, reason in excluded_strategy_mapping.items():
    register_strategy(annotation, None, "excluded", "manual_project_rule", reason)
for annotation, action in approved_manual_strategy_mapping.items():
    register_strategy(annotation, action, "manual_approved", "manual_user_rule")

def map_pedagogical_strategy(record):
    original = record["z_what"]
    decision = strategy_rules.get(normalise_strategy_text(original), {
        "z_what_clean": None,
        "mapping_status": "unmapped",
        "mapping_source": "unmapped_annotation",
        "z_what_exclusion_reason": "unmapped",
    })
    status = decision["mapping_status"]
    return {
        "z_what_original": original,
        **decision,
        "use_for_planner_eval_strict": status == "standard",
        "use_for_planner_eval_extended": status in {"standard", "manual_approved", "provisional"},
        "use_for_diagnosis_eval": bool(record["include_for_evaluation"]),
    }

### 3.2 Apply Strategy Mappings

Add planner fields to a new `planner_annotated_dataset` without removing any of the 669 source records. Preserve the analyser's `exclusion_reason` field; planner-specific exclusions go in **`z_what_exclusion_reason`**. Approved manual CORRECT mappings have no planner exclusion reason.

`planner_strict_dataset` and `planner_extended_dataset` each retain train/validation/test splits. Approved manual and provisional mappings remain distinguishable from standard mappings in the extended subset.


In [3]:
planner_added_features = {
    "z_what_original": Value("string"),
    "z_what_clean": Value("string"),
    "mapping_status": Value("string"),
    "mapping_source": Value("string"),
    "z_what_exclusion_reason": Value("string"),
    "use_for_planner_eval_strict": Value("bool"),
    "use_for_planner_eval_extended": Value("bool"),
    "use_for_diagnosis_eval": Value("bool"),
}
planner_annotated_dataset = DatasetDict()
for split in planner_split_names:
    assert not (set(planner_added_features) & set(cleaned_dataset[split].column_names))
    planner_annotated_dataset[split] = cleaned_dataset[split].map(
        map_pedagogical_strategy,
        features=Features({**cleaned_dataset[split].features, **planner_added_features}),
        keep_in_memory=True, load_from_cache_file=False,
        desc=f"Map planner actions: {split}",
    )

def planner_subset(predicate):
    return DatasetDict({
        split: planner_annotated_dataset[split].filter(
            predicate, keep_in_memory=True, load_from_cache_file=False,
        ) for split in planner_split_names
    })

planner_strict_dataset = planner_subset(lambda row: row["use_for_planner_eval_strict"])
planner_extended_dataset = planner_subset(lambda row: row["use_for_planner_eval_extended"])

### 3.3 Inspect Strategy Mapping Results

Show the original annotations, mapped actions (including unassigned rows), mapping statuses, split counts, and planner-specific exclusions. The main benchmark is strict. Extended results add the two user-approved `CORRECT` mappings and 16 provisional `HINT` mappings. These sources remain distinguishable.


In [4]:
planner_all_records = [
    row for split in planner_split_names for row in planner_annotated_dataset[split]
]
planner_audit_columns = [
    "c_id", "original_split", "source_row_index", "z_what_original", "z_what_clean",
    "mapping_status", "mapping_source", "z_what_exclusion_reason",
    "use_for_planner_eval_strict", "use_for_planner_eval_extended", "use_for_diagnosis_eval",
]
planner_audit = pd.DataFrame([
    {column: row[column] for column in planner_audit_columns}
    for row in planner_all_records
], columns=planner_audit_columns)

original_strategy_distribution = (
    planner_audit["z_what_original"].value_counts(dropna=False)
    .rename_axis("z_what_original").rename("counts").to_frame()
)
print("Original z_what distribution in the 669 source records:")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(original_strategy_distribution)

planner_action_distribution = pd.DataFrame({
    "strict": [
        sum(row["z_what_clean"] == action and row["use_for_planner_eval_strict"]
            for row in planner_all_records) for action in planner_actions
    ],
    "manual_approved_only": [
        sum(row["z_what_clean"] == action and row["mapping_status"] == "manual_approved"
            for row in planner_all_records) for action in planner_actions
    ],
    "provisional_only": [
        sum(row["z_what_clean"] == action and row["mapping_status"] == "provisional"
            for row in planner_all_records) for action in planner_actions
    ],
    "extended": [
        sum(row["z_what_clean"] == action and row["use_for_planner_eval_extended"]
            for row in planner_all_records) for action in planner_actions
    ],
}, index=pd.Index(planner_actions, name="z_what_clean"))
print("Seven-action distribution:")
display(planner_action_distribution)
unassigned_action_count = int(planner_audit["z_what_clean"].isna().sum())
print("Unassigned z_what_clean (None):", unassigned_action_count)

planner_status_counts = (
    planner_audit["mapping_status"].value_counts()
    .reindex(planner_mapping_statuses, fill_value=0)
    .rename_axis("mapping_status").rename("counts").to_frame()
)
print("Mapping statuses:")
display(planner_status_counts)
planner_exclusion_counts = (
    planner_audit["z_what_exclusion_reason"].dropna().value_counts()
    .rename_axis("z_what_exclusion_reason").rename("counts").to_frame()
)
print("Planner exclusion reasons:")
display(planner_exclusion_counts)

Original z_what distribution in the 669 source records:


,counts
z_what_original,
ask_question,204
explain_concept,136
provide_hint,63
provide_strategy,56
provide_correction,40
simplify_question,37
provide_similar_problem,36
affirm_correct_answer,28
encourage_student,22


Seven-action distribution:


,strict,manual_approved_only,provisional_only,extended
z_what_clean,,,,
PROBE,204,0,0,204
HINT,63,0,16,79
EXPLAIN,212,0,0,212
SIMPLIFY,73,0,0,73
CORRECT,40,2,0,42
AFFIRM,28,0,0,28
ENCOURAGE,22,0,0,22


Unassigned z_what_clean (None): 9
Mapping statuses:


,counts
mapping_status,
standard,642
manual_approved,2
provisional,16
excluded,9
unmapped,0


Planner exclusion reasons:


,counts
z_what_exclusion_reason,
non_pedagogical,8
tool_assistance,1


In [5]:
planner_split_summary = pd.DataFrame([
    {
        "split": split,
        "annotated": len(planner_annotated_dataset[split]),
        "diagnosis_eligible": sum(planner_annotated_dataset[split]["use_for_diagnosis_eval"]),
        "strict": len(planner_strict_dataset[split]),
        "extended": len(planner_extended_dataset[split]),
        "provisional": sum(row["mapping_status"] == "provisional"
                           for row in planner_annotated_dataset[split]),
        "excluded": sum(row["mapping_status"] == "excluded"
                        for row in planner_annotated_dataset[split]),
        "manual_approved": sum(row["mapping_status"] == "manual_approved"
                               for row in planner_annotated_dataset[split]),
        "unmapped": sum(row["mapping_status"] == "unmapped"
                        for row in planner_annotated_dataset[split]),
    }
    for split in planner_split_names
]).set_index("split")
display(planner_split_summary)
print("Actions by split (all non-null mappings, including provisional):")
planner_actions_by_split = pd.crosstab(
    planner_audit["z_what_clean"], planner_audit["original_split"]
).reindex(index=planner_actions, columns=planner_split_names, fill_value=0)
display(planner_actions_by_split)

planner_unmapped_annotations = (
    planner_audit.loc[planner_audit["mapping_status"].eq("unmapped"), "z_what_original"]
    .value_counts(dropna=False).rename_axis("unexpected_annotation")
    .rename("counts").reset_index()
)
if planner_unmapped_annotations.empty:
    print("No unexpected unmapped strategy annotations.")
else:
    print("WARNING: Unexpected annotations require clarification; no action was guessed.")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(planner_unmapped_annotations)

print("Provisional mappings requiring manual confirmation:")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(planner_audit.loc[planner_audit["mapping_status"].eq("provisional"), [
        "c_id", "original_split", "source_row_index", "z_what_original", "z_what_clean", "mapping_status"
    ]])

,annotated,diagnosis_eligible,strict,extended,provisional,excluded,manual_approved,unmapped
split,,,,,,,,
train,402,402,383,397,13,5,1,0
validation,69,69,66,68,1,1,1,0
test,198,198,193,195,2,3,0,0


Actions by split (all non-null mappings, including provisional):


original_split,train,validation,test
z_what_clean,,,
PROBE,120,23,61
HINT,54,6,19
EXPLAIN,130,22,60
SIMPLIFY,46,8,19
CORRECT,19,3,20
AFFIRM,15,3,10
ENCOURAGE,13,3,6


No unexpected unmapped strategy annotations.
Provisional mappings requiring manual confirmation:


,c_id,original_split,source_row_index,z_what_original,z_what_clean,mapping_status
11,2979589_63,train,11,Provided a visual aid,HINT,provisional
33,6207151_25,train,33,Provided a visual aid,HINT,provisional
39,2949308_17,train,39,Provided a visual aid,HINT,provisional
73,6136458_53,train,74,Provided a visual aid.,HINT,provisional
100,4104715_98,train,103,Provided a visual aid,HINT,provisional
122,6136280_26,train,125,Provided a visual aid.,HINT,provisional
140,4111721_89,train,143,Provided a visual aid,HINT,provisional
180,2898670_106,train,185,Provided a visual aid,HINT,provisional
193,4104715_77,train,198,Provided a visual aid.,HINT,provisional
223,6383453_36,train,232,Provided a resource,HINT,provisional


In [6]:
approved_manual_case_records = [
    row for row in planner_all_records if row["mapping_status"] == "manual_approved"
]
print("User-approved CORRECT mappings:")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([
        {column: row[column] for column in [
            "c_id", "original_split", "source_row_index", "z_what_original",
            "z_what_clean", "mapping_status", "mapping_source",
        ]}
        for row in approved_manual_case_records
    ]))


User-approved CORRECT mappings:


,c_id,original_split,source_row_index,z_what_original,z_what_clean,mapping_status,mapping_source
0,4294482_65,train,25,Told the student the answer. I don't think it's too important to derive how there are 24 hours in a day. It seems like something you just need to know.,CORRECT,manual_approved,manual_user_rule
1,4111828_8,validation,51,"Told the student the value of pi rounded to 2 decimal places. The student needs to know this if he is to round pi, which I think may be the followup question",CORRECT,manual_approved,manual_user_rule


### 3.4 Validate Strategy Mappings

Verify all 669 records, original fields and splits, diagnosis eligibility, action membership, source-row uniqueness, and subset membership. Compare actual source-label frequencies with those supplied in the instructions. If those frequencies match, require the expected strict (642) and extended (660) totals. The extended total includes the two newly approved CORRECT mappings. Discrepancies are reported explicitly rather than used to fabricate counts.


In [7]:
planner_validation_passed = False
planner_warnings = []
expected_standard_strategy_counts = {
    "ask_question": 204, "provide_hint": 63, "explain_concept": 136,
    "provide_strategy": 56, "provide_example": 20, "simplify_question": 37,
    "provide_similar_problem": 36, "provide_correction": 40,
    "affirm_correct_answer": 28, "encourage_student": 22,
}
expected_nonstandard_strategy_counts = {
    "Provided a visual aid": 8,
    "Provided a visual aid.": 6,
    "Provided a visual aid. The student should then be able to see where the thousands place is.": 1,
    "Provided a resource": 1,
    "Allowed the use of a calculator": 1,
    "Asked the student to fill out the feedback form.": 1,
    "Reminded the student to fill out the feedback form.": 1,
    "Explained the online interface": 1,
    "Said goodbye to the student": 5,
    **{annotation: 1 for annotation in approved_manual_strategy_mapping},
}
actual_strategy_counts = Counter(normalise_strategy_text(row["z_what"]) for row in planner_all_records)
expected_strategy_counts = {
    normalise_strategy_text(label): count
    for label, count in {**expected_standard_strategy_counts, **expected_nonstandard_strategy_counts}.items()
}
planner_count_comparison = pd.DataFrame([
    {
        "normalised_z_what": label,
        "expected": expected_strategy_counts.get(label, 0),
        "actual": actual_strategy_counts.get(label, 0),
        "difference": actual_strategy_counts.get(label, 0) - expected_strategy_counts.get(label, 0),
    }
    for label in sorted(set(expected_strategy_counts) | set(actual_strategy_counts), key=str)
])
planner_count_discrepancies = planner_count_comparison.loc[planner_count_comparison["difference"].ne(0)]
if not planner_count_discrepancies.empty:
    planner_warnings.append("Source strategy frequencies differ from the supplied instructions.")
    with pd.option_context("display.max_colwidth", None):
        display(planner_count_discrepancies)

planner_annotated_count = sum(len(split) for split in planner_annotated_dataset.values())
planner_strict_count = sum(len(split) for split in planner_strict_dataset.values())
planner_extended_count = sum(len(split) for split in planner_extended_dataset.values())
assert planner_annotated_count == 669
assert {split: len(cleaned_dataset[split]) for split in planner_split_names} == {
    "train": 402, "validation": 69, "test": 198,
}
standard_counts_match = all(
    actual_strategy_counts[normalise_strategy_text(label)] == expected
    for label, expected in expected_standard_strategy_counts.items()
)
provisional_counts_match = all(
    actual_strategy_counts[normalise_strategy_text(label)] == expected_nonstandard_strategy_counts[label]
    for label in provisional_strategy_mapping
)
if standard_counts_match:
    assert planner_strict_count == 642
else:
    planner_warnings.append(f"Strict source counts differ: calculated strict size is {planner_strict_count}.")
manual_counts_match = all(
    actual_strategy_counts[normalise_strategy_text(annotation)] == 1
    for annotation in approved_manual_strategy_mapping
)
if standard_counts_match and provisional_counts_match and manual_counts_match:
    assert planner_extended_count == 660
else:
    planner_warnings.append(f"Extended source counts differ: calculated extended size is {planner_extended_count}.")

In [8]:
assert cleaned_dataset is planner_source_reference
assert globals().get("dataset") is planner_raw_reference
assert set(planner_annotated_dataset) == set(planner_strict_dataset) == set(planner_extended_dataset) == set(planner_split_names)
all_source_keys = set()
all_annotated_keys = set()
original_signatures = []
for split in planner_split_names:
    originals = planner_source_rows[split]
    assert list(cleaned_dataset[split]) == originals
    assert cleaned_dataset[split].features == planner_source_features[split]
    assert len(planner_annotated_dataset[split]) == len(originals)
    for column, feature in planner_source_features[split].items():
        assert planner_annotated_dataset[split].features[column] == feature
    for original, mapped in zip(originals, planner_annotated_dataset[split]):
        key = (split, original["source_row_index"])
        assert key not in all_source_keys
        all_source_keys.add(key)
        assert mapped["original_split"] == original["original_split"] == split
        mapped_key = (mapped["original_split"], mapped["source_row_index"])
        assert mapped_key not in all_annotated_keys
        all_annotated_keys.add(mapped_key)
        assert {column: mapped[column] for column in original} == original
        assert mapped["z_what_original"] == mapped["z_what"] == original["z_what"]
        assert {column: mapped[column] for column in planner_added_features} == map_pedagogical_strategy(original)
        assert mapped["mapping_status"] in planner_mapping_statuses
        assert mapped["z_what_clean"] is None or mapped["z_what_clean"] in planner_actions
        assert mapped["use_for_diagnosis_eval"] == original["include_for_evaluation"]
        if mapped["mapping_status"] in {"excluded", "unmapped"}:
            assert mapped["use_for_diagnosis_eval"] is True
            assert not mapped["use_for_planner_eval_strict"]
            assert not mapped["use_for_planner_eval_extended"]
        original_signatures.append(json.dumps(original, sort_keys=True, ensure_ascii=False))
    for subset, eligibility in (
        (planner_strict_dataset, lambda row: row["use_for_planner_eval_strict"]),
        (planner_extended_dataset, lambda row: row["use_for_planner_eval_extended"]),
    ):
        assert list(subset[split]) == [row for row in planner_annotated_dataset[split] if eligibility(row)]
    strict_keys = {row["source_row_index"] for row in planner_strict_dataset[split]}
    extended_keys = {row["source_row_index"] for row in planner_extended_dataset[split]}
    assert strict_keys <= extended_keys
assert all_source_keys == all_annotated_keys
assert len(original_signatures) == len(set(original_signatures)) == planner_annotated_count
assert int(planner_status_counts["counts"].sum()) == planner_annotated_count
assert int(planner_action_distribution["strict"].sum()) == planner_strict_count
assert int(planner_action_distribution["extended"].sum()) == planner_extended_count

# Confirm the case identities supplied for resource and calculator assistance.
for annotation, expected_case, expected_split in (
    ("Provided a resource", "6383453_36", "train"),
    ("Allowed the use of a calculator", "2949298_23", "test"),
):
    matches = [(row["c_id"], row["original_split"]) for row in planner_all_records
               if normalise_strategy_text(row["z_what"]) == normalise_strategy_text(annotation)]
    if matches != [(expected_case, expected_split)]:
        planner_warnings.append(f"Case identity differs for {annotation!r}: {matches!r}.")
if not planner_unmapped_annotations.empty:
    planner_warnings.append("Unexpected strategy annotations remain unmapped and require clarification.")
planner_validation_passed = True
print(f"Validated {planner_annotated_count} records: strict={planner_strict_count}, extended={planner_extended_count}.")
if planner_warnings:
    for warning in planner_warnings:
        print("WARNING:", warning)
else:
    print("No source-count, case-identity, or mapping discrepancies.")
assert {(row["c_id"], row["original_split"]) for row in approved_manual_case_records} == {
    ("4294482_65", "train"), ("4111828_8", "validation"),
}
assert len(approved_manual_case_records) == 2
for row in approved_manual_case_records:
    assert row["z_what_clean"] == "CORRECT"
    assert row["mapping_source"] == "manual_user_rule"
    assert row["z_what_exclusion_reason"] is None
    assert row["use_for_planner_eval_extended"] is True
    assert row["use_for_planner_eval_strict"] is False
print("Both factual-assistance annotations are approved as CORRECT; no cases await a decision.")
print(f"Provisional mappings for confirmation: {int(planner_status_counts.loc['provisional', 'counts'])}.")

Validated 669 records: strict=642, extended=660.
No source-count, case-identity, or mapping discrepancies.
Both factual-assistance annotations are approved as CORRECT; no cases await a decision.
Provisional mappings for confirmation: 16.


### 3.5 Build the 660-Record Retained Dataset

The retained train/validation/test datasets contain only records with **both** a valid `e_clean` and a mapped `z_what_clean`. This includes the 16 provisional HINT mappings and the two manually approved CORRECT mappings. The nine unmapped-to-action calculator/non-pedagogical records join the 31 earlier exclusions in one compilation.

Remove the three evaluation flags and five processing-control fields from the final retained records. Intermediate audit variables above keep those fields for reproducibility; they are not part of the shared split-file schema. All remaining original fields, labels, source indices, and split assignments are preserved.

At this stage, the four intermediate exports are `bridge_train.jsonl`, `bridge_validation.jsonl`, `bridge_test.jsonl`, and `bridge_excluded.jsonl`. Only the excluded compilation keeps `exclusion_reason`, plus `exclusion_stage`, to explain why each record was omitted. There is no separate strict/extended export step. The earlier error-cleaning export is an intermediate output; this stage combines exclusions. Segment 4 adds cleaned topic names, and Segment 6 writes the final 12-field schema.


In [9]:
assert planner_validation_passed, "Complete planner validation before exporting."

# Final shared files contain only successfully mapped error and strategy labels.
final_error_labels = {
    "guess", "careless", "right-idea", "misinterpret", "diagnose", "imprecise", "no_error",
}
removed_final_fields = {
    "use_for_planner_eval_strict", "use_for_planner_eval_extended", "use_for_diagnosis_eval",
    "exclusion_reason", "include_for_evaluation", "z_what_exclusion_reason",
    "mapping_status", "mapping_source",
}
final_columns = [
    column for column in planner_annotated_dataset["train"].column_names
    if column not in removed_final_fields
]

def has_both_mapped_labels(record):
    return record["e_clean"] in final_error_labels and record["z_what_clean"] in planner_actions

final_dataset = DatasetDict({
    split: planner_annotated_dataset[split].filter(
        has_both_mapped_labels,
        keep_in_memory=True, load_from_cache_file=False,
        desc=f"Keep both mapped labels: {split}",
    ).remove_columns(list(removed_final_fields))
    for split in planner_split_names
})

def excluded_audit_record(record, stage, reason):
    """Give the combined exclusion file one consistent audit schema."""
    strategy_decision = map_pedagogical_strategy(record)
    result = {column: record.get(column) for column in final_columns}
    result["z_what_original"] = record["z_what"]
    result["z_what_clean"] = strategy_decision["z_what_clean"]
    result["exclusion_stage"] = stage
    result["exclusion_reason"] = reason
    return result

# Preserve earlier e-cleaning exclusions, including the removed train duplicate.
initial_excluded_records = [
    row for split in planner_split_names for row in excluded_dataset[split]
]
new_strategy_excluded_records = [
    row for split in planner_split_names for row in planner_annotated_dataset[split]
    if not has_both_mapped_labels(row)
]
final_excluded_records = [
    excluded_audit_record(row, "initial_cleaning", row["exclusion_reason"])
    for row in initial_excluded_records
] + [
    excluded_audit_record(
        row, "strategy_mapping", row["z_what_exclusion_reason"] or "unmapped_strategy"
    )
    for row in new_strategy_excluded_records
]

# Check every original source row is present exactly once in a final split or audit.
source_keys = {
    (row["original_split"], row["source_row_index"])
    for split in planner_split_names for row in planner_annotated_dataset[split]
} | {
    (row["original_split"], row["source_row_index"]) for row in initial_excluded_records
}
retained_keys = []
for split in planner_split_names:
    source_by_index = {
        row["source_row_index"]: row for row in planner_annotated_dataset[split]
    }
    assert final_dataset[split].column_names == final_columns
    assert not (removed_final_fields & set(final_dataset[split].column_names))
    for row in final_dataset[split]:
        assert row["original_split"] == split
        assert has_both_mapped_labels(row)
        assert {column: row[column] for column in final_columns} == {
            column: source_by_index[row["source_row_index"]][column]
            for column in final_columns
        }
        retained_keys.append((split, row["source_row_index"]))
excluded_keys = [
    (row["original_split"], row["source_row_index"]) for row in final_excluded_records
]
assert len(retained_keys) == len(set(retained_keys))
assert len(excluded_keys) == len(set(excluded_keys))
assert set(retained_keys).isdisjoint(excluded_keys)
assert set(retained_keys) | set(excluded_keys) == source_keys
assert len(source_keys) == 700
assert len(retained_keys) + len(excluded_keys) == len(source_keys)
assert all(set(row) == set(final_columns) | {"exclusion_stage", "exclusion_reason"}
           for row in final_excluded_records)

final_split_summary = pd.DataFrame([
    {
        "split": split,
        "retained": len(final_dataset[split]),
        "excluded": sum(row["original_split"] == split for row in final_excluded_records),
    }
    for split in planner_split_names
]).set_index("split")
print("Final shared dataset counts:")
display(final_split_summary)
print("Final retained fields:", final_columns)
final_action_distribution = pd.Series(Counter(
    row["z_what_clean"] for split in planner_split_names for row in final_dataset[split]
)).reindex(planner_actions, fill_value=0).rename_axis("z_what_clean").rename("counts").to_frame()
display(final_action_distribution)
print("Combined exclusion reasons:")
display(pd.Series(Counter(row["exclusion_reason"] for row in final_excluded_records))
        .rename_axis("exclusion_reason").rename("counts").to_frame())

final_output_directory = Path("data")
final_output_directory.mkdir(parents=True, exist_ok=True)

def export_final_jsonl(path, records):
    records = list(records)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
    with path.open(encoding="utf-8") as handle:
        saved = [json.loads(line) for line in handle]
    assert saved == records
    return {"file": path.as_posix(), "records": len(records)}

final_export_manifest = [
    export_final_jsonl(final_output_directory / f"bridge_{split}.jsonl", final_dataset[split])
    for split in planner_split_names
]
final_export_manifest.append(export_final_jsonl(
    final_output_directory / "bridge_excluded.jsonl", final_excluded_records
))
print("Validated and exported the four final shared files:")
display(pd.DataFrame(final_export_manifest))


Final shared dataset counts:


,retained,excluded
split,,
train,397,22
validation,68,3
test,195,15


Final retained fields: ['z_why', 'c_id', 'c_h', 'c_r_', 'c_r', 'lesson_topic', 'c_revision', 'z_what', 'e', 'e_original', 'e_clean', 'original_split', 'source_row_index', 'z_what_original', 'z_what_clean']


,counts
z_what_clean,
PROBE,204
HINT,79
EXPLAIN,212
SIMPLIFY,73
CORRECT,42
AFFIRM,28
ENCOURAGE,22


Combined exclusion reasons:


,counts
exclusion_reason,
non_diagnostic,20
ambiguous,7
duplicate_train_test,1
tutor_error,3
non_pedagogical,8
tool_assistance,1


Validated and exported the four final shared files:


,file,records
0,data/bridge_train.jsonl,397
1,data/bridge_validation.jsonl,68
2,data/bridge_test.jsonl,195
3,data/bridge_excluded.jsonl,40


<a id="segment-4"></a>

## Segment 4: Lesson Topic Standardisation

### 4.1 Count Original Lesson Topics

Count exact original `lesson_topic` values in the 660 retained records before structural topic cleaning. Original names remain available after cleaning for traceability.


In [1]:
from collections import Counter

lesson_topic_counts = Counter(
    topic
    for split in final_dataset.values()
    for topic in split["lesson_topic"]
)
unique_lesson_topics = sorted(lesson_topic_counts)

print("Number of unique lesson topics:", len(unique_lesson_topics))
print("lesson_topic | count")
print("-" * 80)
for topic in unique_lesson_topics:
    print(f"{topic} | {lesson_topic_counts[topic]}")
print("Total records:", sum(lesson_topic_counts.values()))


Number of unique lesson topics: 203
lesson_topic | count
--------------------------------------------------------------------------------
1.2C.Place Value (Review) - 1 | 1
1.3B.Addition & Subtraction Properties (Review) - 1 | 4
1.7A.Length | 1
2.2A.Place Value | 7
2.2B.Expanded Notation (Review) - 1 | 1
2.2C.Counting Numbers | 1
2.4A.Addition and Subtraction Facts | 1
2.4B.Addition and Subtraction Within 100 | 3
2.4C.Money (Review) - 1 | 2
2.6A.Foundations of Multiplication | 3
2.8A.2- and 3-Dimensional Shapes | 20
2.8A.2- and 3-Dimensional Shapes (Review) - 1 | 2
2.9A.Length Problems | 7
2.9A.Length Problems (Review) - 1 | 4
2.9A.Measuring Length | 6
2.9F.Area by Counting | 1
2.9F.Area by Counting (Review) - 2 | 1
2.9G.Time (Review) - 1 | 2
3.1A.Bar Graphs | 3
3.1G.Shapes and Area | 5
3.1G.Shapes and Area (Review) - 1 | 2
3.1G.Shapes and Area (Review) - 2 | 2
3.2B.Multiples of Ten | 8
3.3A.Fraction Concepts | 5
3.3B.Fractions on a Number Line (Review) - 1 | 2
3.3E.Bar Graphs | 1
3.3F.

### 4.2 Inspect the Approved Multi-Topic Cases

Reuse `final_dataset`, the 660 retained records (397 train, 68 validation, 195 test). The review dataset below contains only the semicolon-separated multi-topic records and preserves every original column and split. It temporarily adds `lesson_topic_clean = None` and `lesson_topic_needs_review = True`.

The histories were inspected and both assignments are now approved: test row 76 becomes Shapes and Area; train row 115 becomes Graphing Points on the Coordinate Plane. Section 4.3 defines these explicit decisions. No topic is inferred from the expert responses. This review stage does not alter `final_dataset` or the shared JSONL exports.


In [1]:
from collections import Counter
from copy import deepcopy
import pandas as pd
from datasets import DatasetDict, Features, Value
from IPython.display import display, Markdown

if "final_dataset" not in globals():
    raise RuntimeError("Run the final shared dataset section before reviewing lesson topics.")
lesson_topic_source_reference = final_dataset
lesson_topic_source_snapshot = {
    split: deepcopy(list(records)) for split, records in final_dataset.items()
}
assert {split: len(records) for split, records in final_dataset.items()} == {
    "train": 397, "validation": 68, "test": 195,
}
multitopic_values_for_review = (
    "4.2D.Rounding;3.6E.Shapes and Area",
    "6.4E.Understanding Percents;6.2C.Graphing Points on the Coordinate Plane",
)
all_multitopic_records = [
    record for split in final_dataset.values() for record in split
    if isinstance(record["lesson_topic"], str) and ";" in record["lesson_topic"]
]
multitopic_counts = Counter(record["lesson_topic"] for record in all_multitopic_records)
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([
        {"lesson_topic": topic, "count": multitopic_counts[topic]}
        for topic in multitopic_values_for_review
    ]))
unexpected_multitopic_values = set(multitopic_counts) - set(multitopic_values_for_review)
if unexpected_multitopic_values:
    print("Additional multi-topic values also require manual review:")
    display(pd.DataFrame([
        {"lesson_topic": topic, "count": multitopic_counts[topic]}
        for topic in sorted(unexpected_multitopic_values)
    ]))

multi_topic_review_dataset = DatasetDict()
for split, records in final_dataset.items():
    selected = records.filter(
        lambda row: isinstance(row["lesson_topic"], str) and ";" in row["lesson_topic"],
        keep_in_memory=True, load_from_cache_file=False,
    )
    # Explicit nullable string features also support empty splits.
    review_features = Features({
        **records.features,
        "lesson_topic_clean": Value("string"),
        "lesson_topic_needs_review": Value("bool"),
    })
    if len(selected):
        selected = selected.map(
            lambda row: {"lesson_topic_clean": None, "lesson_topic_needs_review": True},
            features=review_features,
            keep_in_memory=True, load_from_cache_file=False,
        )
    else:
        from datasets import Dataset
        selected = Dataset.from_dict(
            {column: [] for column in review_features}, features=review_features
        )
    multi_topic_review_dataset[split] = selected

,lesson_topic,count
0,4.2D.Rounding;3.6E.Shapes and Area,1
1,6.4E.Understanding Percents;6.2C.Graphing Points on the Coordinate Plane,1


In [2]:
for topic in (*multitopic_values_for_review, *sorted(unexpected_multitopic_values)):
    matching_records = [
        record for split in multi_topic_review_dataset.values()
        for record in split if record["lesson_topic"] == topic
    ]
    display(Markdown(f"#### {topic} ({len(matching_records)} record(s))"))
    for record in matching_records:
        with pd.option_context("display.max_rows", None, "display.max_columns", None,
                               "display.max_colwidth", None):
            display(pd.DataFrame([{
                column: record[column] for column in [
                    "c_id", "original_split", "source_row_index", "lesson_topic",
                    "lesson_topic_clean", "lesson_topic_needs_review",
                ]
            }]))
            print("Complete conversation history (c_h):")
            # Every original message field is shown; no turns are truncated.
            display(pd.DataFrame(record["c_h"]))
            print("Optional expert revised response (c_r_) ? manual review only:")
            display(pd.DataFrame(record["c_r_"]))

assert final_dataset is lesson_topic_source_reference
for split, records in final_dataset.items():
    assert list(records) == lesson_topic_source_snapshot[split]
for split, records in multi_topic_review_dataset.items():
    for record in records:
        assert record["original_split"] == split
        assert record["lesson_topic_clean"] is None
        assert record["lesson_topic_needs_review"] is True
print("Initial review complete. Both approved assignments are applied in Step 2 below.")

#### 4.2D.Rounding;3.6E.Shapes and Area (1 record(s))

,c_id,original_split,source_row_index,lesson_topic,lesson_topic_clean,lesson_topic_needs_review
0,2915943_61,test,76,4.2D.Rounding;3.6E.Shapes and Area,None,True


Complete conversation history (c_h):


,id,text,time,user
0,61,Pablo was asked to divide shapes into 8 equal parts.,29:22,tutor
1,62,Option A is the correct answer.,30:56,tutor
2,63,Did you get it?,31:28,tutor
3,64,yes,31:41,student


Optional expert revised response (c_r_) ? manual review only:


...

Complete conversation history (c_h):


,id,text,time,user
0,173,What is the x-axis value?,56:28,tutor
1,174,"Sorry, we are running out of time.",57:20,tutor
2,175,What is the final answer to this question?,57:22,tutor
3,176,-4 5,57:45,student


Optional expert revised response (c_r_) ? manual review only:


,is_revised,text,user
0,True,"Ah, we may have gotten our x and y coordinates mixed up",tutor
1,True,"Remember that the first coordinate is the x, which tells us how far left/right from the center we go",tutor
2,True,"The second coordinate is the y, which tells us how far up/down from the center we go",tutor
3,True,"Looking at the dot, how far left/right from the center is it?",tutor


Initial review complete. Both approved assignments are applied in Step 2 below.


### 4.3 Define Topic Rules and Approved Assignments

Remove recognised curriculum codes from the beginning, review identifiers from the end, and unnecessary whitespace. Preserve mathematical numbers and the remaining topic wording. The observed `(Review) - 1.` ending is also treated as a review identifier. Only exactly identical cleaned strings are grouped; related topics are not semantically merged.

| Case | Split | Source row | User-approved topic |
|---|---|---:|---|
| `2915943_61` | test | 76 | Shapes and Area |
| `4273568_173` | train | 115 | Graphing Points on the Coordinate Plane |

New multi-topic records without explicit approval stay unassigned and stop export. The user-approved spelling correction `Multipying Mixed Numbers` to `Multiplying Mixed Numbers` applies only to `lesson_topic_clean`; original topic text stays unchanged. Other spelling concerns require separate approval.


In [1]:
import re
import json
from collections import Counter
from copy import deepcopy
from pathlib import Path
import pandas as pd
from datasets import DatasetDict, Features, Value
from IPython.display import display

# Covers 1.2C., A2.7D., and 7.NS.1b.; does not consume the readable topic.
curriculum_prefix_pattern = re.compile(
    r"^(?:[A-Za-z]*\d+)\.(?:[A-Za-z]+\.)?\d+[A-Za-z]*\.\s*"
)
review_suffix_pattern = re.compile(
    r"\s*\(\s*Review\s*\)\s*(?:-\s*\d+\s*\.?)?\s*$",
    flags=re.IGNORECASE,
)

approved_topic_spelling_corrections = {
    "Multipying Mixed Numbers": "Multiplying Mixed Numbers",
}

def clean_single_lesson_topic(value):
    """Remove structural identifiers while preserving the mathematical wording."""
    if not isinstance(value, str) or not value.strip() or ";" in value:
        return None
    topic = " ".join(value.split())
    topic = curriculum_prefix_pattern.sub("", topic, count=1)
    topic = review_suffix_pattern.sub("", topic)
    topic = " ".join(topic.split())
    return approved_topic_spelling_corrections.get(topic, topic) or None

manual_topic_assignments = {
    ("test", 76, "2915943_61"): "Shapes and Area",
    ("train", 115, "4273568_173"): "Graphing Points on the Coordinate Plane",
}
expected_multitopic_annotations = {
    ("test", 76, "2915943_61"): "4.2D.Rounding;3.6E.Shapes and Area",
    ("train", 115, "4273568_173"): (
        "6.4E.Understanding Percents;6.2C.Graphing Points on the Coordinate Plane"
    ),
}

def clean_lesson_topic_record(record):
    raw_topic = record["lesson_topic"]
    key = (record["original_split"], record["source_row_index"], record["c_id"])
    if key in manual_topic_assignments:
        assert raw_topic == expected_multitopic_annotations[key]
        chosen = manual_topic_assignments[key]
        assert chosen in [clean_single_lesson_topic(part) for part in raw_topic.split(";")]
        return {"lesson_topic_clean": chosen}
    return {"lesson_topic_clean": clean_single_lesson_topic(raw_topic)}

### 4.4 Apply Topic Cleaning and Inspect Frequencies

Reuse `final_dataset`, adding only `lesson_topic_clean` to a new `lesson_topic_cleaned_dataset`. All original columns, records, order, and split assignments are preserved. Rerunning this step recomputes only the cleaned-topic column.


In [2]:
lesson_topic_input_dataset = DatasetDict({
    split: records.remove_columns("lesson_topic_clean")
    if "lesson_topic_clean" in records.column_names else records
    for split, records in final_dataset.items()
})
lesson_topic_before_rows = {
    split: deepcopy(list(records)) for split, records in lesson_topic_input_dataset.items()
}
lesson_topic_cleaned_dataset = DatasetDict({
    split: records.map(
        clean_lesson_topic_record,
        features=Features({**records.features, "lesson_topic_clean": Value("string")}),
        keep_in_memory=True, load_from_cache_file=False,
        desc=f"Standardise lesson topics: {split}",
    )
    for split, records in lesson_topic_input_dataset.items()
})
lesson_topic_audit = pd.DataFrame([
    {column: record[column] for column in [
        "c_id", "original_split", "source_row_index", "lesson_topic", "lesson_topic_clean"
    ]}
    for split in lesson_topic_cleaned_dataset.values() for record in split
])
lesson_topic_mapping_counts = (
    lesson_topic_audit.groupby(["lesson_topic", "lesson_topic_clean"], dropna=False)
    .size().rename("count").reset_index()
)
print("Original-to-cleaned topic mapping and frequencies:")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(lesson_topic_mapping_counts)
print("Approved multi-topic assignments:")
with pd.option_context("display.max_colwidth", None):
    display(lesson_topic_audit.loc[lesson_topic_audit["lesson_topic"].str.contains(";", regex=False)])

unresolved_lesson_topics = lesson_topic_audit.loc[lesson_topic_audit["lesson_topic_clean"].isna()]
if not unresolved_lesson_topics.empty:
    print("WARNING: Resolve these topic labels before export:")
    with pd.option_context("display.max_colwidth", None):
        display(unresolved_lesson_topics)

print("Applied approved spelling correction:")
applied_topic_spelling_corrections = lesson_topic_mapping_counts.loc[
    lesson_topic_mapping_counts["lesson_topic"].eq("5.3K.Multipying Mixed Numbers")
].copy()
with pd.option_context("display.max_colwidth", None):
    display(applied_topic_spelling_corrections)

cleaned_lesson_topic_counts = (
    lesson_topic_audit["lesson_topic_clean"].value_counts()
    .sort_index().rename_axis("lesson_topic_clean").rename("count").to_frame()
)
print("Unique original topics:", lesson_topic_audit["lesson_topic"].nunique())
print("Unique cleaned topics:", len(cleaned_lesson_topic_counts))
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(cleaned_lesson_topic_counts)

Original-to-cleaned topic mapping and frequencies:


,lesson_topic,lesson_topic_clean,count
0,1.2C.Place Value (Review) - 1,Place Value,1
1,1.3B.Addition & Subtraction Properties (Review) - 1,Addition & Subtraction Properties,4
2,1.7A.Length,Length,1
3,2.2A.Place Value,Place Value,7
4,2.2B.Expanded Notation (Review) - 1,Expanded Notation,1
5,2.2C.Counting Numbers,Counting Numbers,1
6,2.4A.Addition and Subtraction Facts,Addition and Subtraction Facts,1
7,2.4B.Addition and Subtraction Within 100,Addition and Subtraction Within 100,3
8,2.4C.Money (Review) - 1,Money,2
9,2.6A.Foundations of Multiplication,Foundations of Multiplication,3


Approved multi-topic assignments:


,c_id,original_split,source_row_index,lesson_topic,lesson_topic_clean
111,4273568_173,train,115,6.4E.Understanding Percents;6.2C.Graphing Points on the Coordinate Plane,Graphing Points on the Coordinate Plane
536,2915943_61,test,76,4.2D.Rounding;3.6E.Shapes and Area,Shapes and Area


Applied approved spelling correction:


,lesson_topic,lesson_topic_clean,count
141,5.3K.Multipying Mixed Numbers,Multiplying Mixed Numbers,2


Unique original topics: 203
Unique cleaned topics: 124


,count
lesson_topic_clean,
2- and 3-Dimensional Shapes,22
Adding Multi-Digit Numbers,7
Adding Rational Numbers,1
Adding Whole Numbers,6
Adding and Subtracting Decimal Numbers,6
Adding and Subtracting Decimals,2
Adding and Subtracting Fractions,1
Adding and Subtracting Mixed Numbers,1
Addition & Subtraction Properties,4


### 4.5 Validate Topics and Export the Intermediate Schema

Test the supplied code formats and preserve numbers inside mathematics topic names. Verify both user-approved assignments, all 660 records, every original field, row order, and split membership. Export only when all topics are non-empty strings. The original `lesson_topic` remains unchanged, and the 40-record exclusion compilation is preserved.


In [3]:
lesson_topic_validation_passed = False
expected_topic_examples = {
    "5.3K.Multipying Mixed Numbers": "Multiplying Mixed Numbers",
    "Multipying Mixed Numbers": "Multiplying Mixed Numbers",
    "1.2C.Place Value (Review) - 1": "Place Value",
    "3.6E.Shapes and Area": "Shapes and Area",
    "3.6E.Shapes and Area (Review) - 1": "Shapes and Area",
    "3.6E.Shapes and Area (Review) - 2": "Shapes and Area",
    "3.4B.Rounding Whole Numbers (Review) - 2": "Rounding Whole Numbers",
    "4.2A.Comparing Decimals": "Comparing Decimals",
    "5.4B.Order of Operations": "Order of Operations",
    "A2.7D.Factoring Polynomials": "Factoring Polynomials",
    "7.NS.1b.Adding Rational Numbers": "Adding Rational Numbers",
    "7.RP.2a.Proportional Relationship Graphs (Review) - 1.": "Proportional Relationship Graphs",
    "2- and 3-Dimensional Shapes": "2- and 3-Dimensional Shapes",
    "2.8A.2- and 3-Dimensional Shapes": "2- and 3-Dimensional Shapes",
    "Multiplying Two-Digit by Two-Digit Numbers": "Multiplying Two-Digit by Two-Digit Numbers",
    " 3.6E.Shapes  and Area (Review) - 1 ": "Shapes and Area",
}
for raw, expected in expected_topic_examples.items():
    assert clean_single_lesson_topic(raw) == expected, raw
assert clean_single_lesson_topic("4.2D.Rounding;3.6E.Shapes and Area") is None
for left, right in [
    ("Adding Whole Numbers", "Adding Multi-Digit Numbers"),
    ("Comparing Decimals", "Comparing Decimal Numbers"),
    ("Area", "Shapes and Area"),
]:
    assert clean_single_lesson_topic(left) != clean_single_lesson_topic(right)
assert {split: len(records) for split, records in lesson_topic_cleaned_dataset.items()} == {
    "train": 397, "validation": 68, "test": 195,
}
assert sum(len(records) for records in lesson_topic_cleaned_dataset.values()) == 660
assert unresolved_lesson_topics.empty, "Resolve pending topics before export."
seen_lesson_topic_keys = set()
applied_manual_topic_keys = set()
for split, records in lesson_topic_cleaned_dataset.items():
    originals = lesson_topic_before_rows[split]
    assert list(lesson_topic_input_dataset[split]) == originals
    assert records.column_names == [*lesson_topic_input_dataset[split].column_names, "lesson_topic_clean"]
    for original, record in zip(originals, records):
        assert {column: record[column] for column in original} == original
        assert record["original_split"] == split
        assert isinstance(record["lesson_topic_clean"], str) and record["lesson_topic_clean"].strip()
        assert ";" not in record["lesson_topic_clean"]
        assert not curriculum_prefix_pattern.match(record["lesson_topic_clean"])
        assert not review_suffix_pattern.search(record["lesson_topic_clean"])
        assert record["lesson_topic_clean"] == clean_lesson_topic_record(original)["lesson_topic_clean"]
        key = (split, record["source_row_index"], record["c_id"])
        assert key not in seen_lesson_topic_keys
        seen_lesson_topic_keys.add(key)
        if key in manual_topic_assignments:
            assert record["lesson_topic_clean"] == manual_topic_assignments[key]
            applied_manual_topic_keys.add(key)
assert applied_manual_topic_keys == set(manual_topic_assignments)
assert len(seen_lesson_topic_keys) == 660
assert int(cleaned_lesson_topic_counts["count"].sum()) == 660
lesson_topic_validation_passed = True
print("Validated all 660 records, original fields, split memberships, and both approved assignments.")

Validated all 660 records, original fields, split memberships, and both approved assignments.


In [4]:
assert lesson_topic_validation_passed, "Validate lesson-topic cleaning before export."
lesson_topic_output_directory = Path("data")
excluded_topic_audit_path = lesson_topic_output_directory / "bridge_excluded.jsonl"
excluded_topic_audit_before = excluded_topic_audit_path.read_bytes()
lesson_topic_export_manifest = []
for split, records in lesson_topic_cleaned_dataset.items():
    output_path = lesson_topic_output_directory / f"bridge_{split}.jsonl"
    with output_path.open("w", encoding="utf-8", newline=chr(10)) as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + chr(10))
    with output_path.open(encoding="utf-8") as handle:
        saved_records = [json.loads(line) for line in handle]
    assert saved_records == list(records)
    lesson_topic_export_manifest.append({"file": output_path.as_posix(), "records": len(saved_records)})
assert excluded_topic_audit_path.read_bytes() == excluded_topic_audit_before
final_dataset = lesson_topic_cleaned_dataset
print("Final split files now include lesson_topic_clean:")
display(pd.DataFrame(lesson_topic_export_manifest))
print("Excluded compilation unchanged: 40 records.")

Final split files now include lesson_topic_clean:


,file,records
0,data/bridge_train.jsonl,397
1,data/bridge_validation.jsonl,68
2,data/bridge_test.jsonl,195


Excluded compilation unchanged: 40 records.


<a id="segment-5"></a>

## Segment 5: Topic Merge Preview

### 5.1 Define and Check the Nine Exact Merges

Apply only the nine exact, user-specified mappings to `lesson_topic_clean`. Unlisted topics and every other field stay unchanged. This preview reuses the final 660-record `final_dataset` and keeps its split assignments.

The candidate result is **`topic_merged_preview_dataset`**. `final_dataset` remains the pre-merge source so the comparison can be rerun consistently. **No JSONL exports are written by this section.** Review the frequency tables and validation results before approving export.


In [1]:
from collections import Counter
from copy import deepcopy
import pandas as pd
from datasets import DatasetDict
from IPython.display import display

TOPIC_MERGES = {
    "Comparing Decimal Numbers": "Comparing Decimals",
    "Adding and Subtracting Decimal Numbers": "Adding and Subtracting Decimals",
    "Classifying Two-Dimensional Figures": "Classifying 2D Figures",
    "Simple Expression": "Simple Expressions",
    "Identify Unit Rate": "Unit Rates",
    "Understanding Unit Rates": "Unit Rates",
    "Divide Two Fractions": "Dividing Fractions",
    "Understanding Equivalent Fractions": "Equivalent Fractions",
    "Solving with Order of Operations": "Order of Operations",
}

if "final_dataset" not in globals():
    raise RuntimeError("Run the final lesson-topic cleaning cells before this preview.")
topic_merge_source_reference = final_dataset
topic_merge_source_rows = {
    split: deepcopy(list(records)) for split, records in final_dataset.items()
}
topic_counts_before_merges = Counter(
    row["lesson_topic_clean"] for records in final_dataset.values() for row in records
)
missing_topic_merge_sources = sorted(set(TOPIC_MERGES) - set(topic_counts_before_merges))
print("Missing source topic names:", missing_topic_merge_sources or "None; all nine exist.")
topic_merge_source_check = pd.DataFrame([
    {
        "source_topic": source,
        "source_count_before": topic_counts_before_merges[source],
        "target_topic": target,
        "target_count_before": topic_counts_before_merges[target],
    }
    for source, target in TOPIC_MERGES.items()
])
with pd.option_context("display.max_colwidth", None):
    display(topic_merge_source_check)
assert len(TOPIC_MERGES) == 9
assert not missing_topic_merge_sources, "Review missing source topics before proceeding."

Missing source topic names: None; all nine exist.


,source_topic,source_count_before,target_topic,target_count_before
0,Comparing Decimal Numbers,8,Comparing Decimals,7
1,Adding and Subtracting Decimal Numbers,6,Adding and Subtracting Decimals,2
2,Classifying Two-Dimensional Figures,1,Classifying 2D Figures,17
3,Simple Expression,3,Simple Expressions,3
4,Identify Unit Rate,5,Unit Rates,2
5,Understanding Unit Rates,1,Unit Rates,2
6,Divide Two Fractions,1,Dividing Fractions,4
7,Understanding Equivalent Fractions,8,Equivalent Fractions,3
8,Solving with Order of Operations,2,Order of Operations,5


### 5.2 Compare Topic Frequencies Before and After Merging

The complete comparison and destination-topic summary describe the preview only.


In [2]:
def merge_approved_lesson_topic(record):
    original_cleaned_topic = record["lesson_topic_clean"]
    return {
        "lesson_topic_clean": TOPIC_MERGES.get(original_cleaned_topic, original_cleaned_topic)
    }

topic_merged_preview_dataset = DatasetDict({
    split: records.map(
        merge_approved_lesson_topic,
        features=records.features,
        keep_in_memory=True, load_from_cache_file=False,
        desc=f"Preview exact topic merges: {split}",
    )
    for split, records in final_dataset.items()
})
topic_counts_after_merges = Counter(
    row["lesson_topic_clean"] for records in topic_merged_preview_dataset.values() for row in records
)
topic_frequency_comparison = pd.DataFrame([
    {
        "lesson_topic_clean": topic,
        "before": topic_counts_before_merges[topic],
        "after": topic_counts_after_merges[topic],
        "difference": topic_counts_after_merges[topic] - topic_counts_before_merges[topic],
    }
    for topic in sorted(set(topic_counts_before_merges) | set(topic_counts_after_merges))
]).set_index("lesson_topic_clean")
print("All topic frequencies before and after merging (zero after means the source was merged):")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(topic_frequency_comparison)

merged_target_frequencies = pd.DataFrame([
    {
        "lesson_topic_clean": target,
        "before": topic_counts_before_merges[target],
        "after": topic_counts_after_merges[target],
    }
    for target in sorted(set(TOPIC_MERGES.values()))
]).set_index("lesson_topic_clean")
print("Final frequencies for the eight destination topics:")
display(merged_target_frequencies)

All topic frequencies before and after merging (zero after means the source was merged):


,before,after,difference
lesson_topic_clean,,,
2- and 3-Dimensional Shapes,22,22,0
Adding Multi-Digit Numbers,7,7,0
Adding Rational Numbers,1,1,0
Adding Whole Numbers,6,6,0
Adding and Subtracting Decimal Numbers,6,0,-6
Adding and Subtracting Decimals,2,8,6
Adding and Subtracting Fractions,1,1,0
Adding and Subtracting Mixed Numbers,1,1,0
Addition & Subtraction Properties,4,4,0


Final frequencies for the eight destination topics:


,before,after
lesson_topic_clean,,
Adding and Subtracting Decimals,2,8
Classifying 2D Figures,17,18
Comparing Decimals,7,15
Dividing Fractions,4,5
Equivalent Fractions,3,11
Order of Operations,5,7
Simple Expressions,3,6
Unit Rates,2,8


### 5.3 Validate the Preview Without Exporting

Check the nine-topic reduction and unchanged records, fields, and split assignments.


In [3]:
topic_merge_preview_validated = False
assert final_dataset is topic_merge_source_reference
assert {split: len(records) for split, records in topic_merged_preview_dataset.items()} == {
    "train": 397, "validation": 68, "test": 195,
}
assert set(topic_merged_preview_dataset) == set(final_dataset)
changed_topic_record_count = 0
seen_topic_merge_keys = set()
for split, records in topic_merged_preview_dataset.items():
    originals = topic_merge_source_rows[split]
    assert list(final_dataset[split]) == originals, "The pre-merge source changed."
    assert records.column_names == final_dataset[split].column_names
    assert records.features == final_dataset[split].features
    for original, candidate in zip(originals, records):
        assert {key: value for key, value in candidate.items() if key != "lesson_topic_clean"} == {
            key: value for key, value in original.items() if key != "lesson_topic_clean"
        }
        assert candidate["lesson_topic_clean"] == TOPIC_MERGES.get(
            original["lesson_topic_clean"], original["lesson_topic_clean"]
        )
        assert candidate["original_split"] == split
        source_key = (split, candidate["source_row_index"])
        assert source_key not in seen_topic_merge_keys
        seen_topic_merge_keys.add(source_key)
        changed_topic_record_count += candidate["lesson_topic_clean"] != original["lesson_topic_clean"]
assert len(seen_topic_merge_keys) == 660
assert sum(topic_counts_before_merges.values()) == sum(topic_counts_after_merges.values()) == 660
assert changed_topic_record_count == sum(topic_counts_before_merges[source] for source in TOPIC_MERGES)
expected_topic_counts_after = Counter()
for topic, count in topic_counts_before_merges.items():
    expected_topic_counts_after[TOPIC_MERGES.get(topic, topic)] += count
assert topic_counts_after_merges == expected_topic_counts_after
assert not (set(TOPIC_MERGES) & set(topic_counts_after_merges))
unique_topics_before_merges = len(topic_counts_before_merges)
unique_topics_after_merges = len(topic_counts_after_merges)
assert unique_topics_before_merges - unique_topics_after_merges == 9
assert (topic_frequency_comparison["after"] - topic_frequency_comparison["before"]).sum() == 0
topic_merge_preview_validated = True
print(f"Validated: {unique_topics_before_merges} -> {unique_topics_after_merges} unique topics (decrease: 9).")
print(f"Updated only lesson_topic_clean in {changed_topic_record_count} records.")
print("All 660 records, split assignments, original lesson_topic, c_h, e_clean, z_what_clean, and other fields are preserved.")
print("Preview ready for review. JSONL exports have not been overwritten.")

Validated: 124 -> 115 unique topics (decrease: 9).
Updated only lesson_topic_clean in 35 records.
All 660 records, split assignments, original lesson_topic, c_h, e_clean, z_what_clean, and other fields are preserved.
Preview ready for review. JSONL exports have not been overwritten.


<a id="segment-6"></a>

## Segment 6: Final Schema and Export

### 6.1 Inspect Source Fields and Annotation Copies

Reuse `final_dataset`, the current 660 cleaned records. This section does not reload BRIDGE or repeat error, strategy, or lesson-topic cleaning. The nine proposed topic merges remain in their separate preview and are not applied here: existing label values are preserved exactly.

Check that `e_original == e` and `z_what_original == z_what` before removing those redundant copies. Inspect every message for missing/unexpected fields, invalid structures, and unexpected roles. Any issue is displayed with the affected case ID and blocks export.

The original histories stay intact in `final_dataset` and the intermediate `history_annotated_dataset`. Before replacing the shared JSONL files, preserve all original 16-field records, including timestamps, `c_r`, and `c_revision`, under `data/audit/before_history_reformat/`.


In [1]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import json
import pandas as pd
from datasets import Dataset, DatasetDict, Features, List, Value
from IPython.display import display

if "final_dataset" not in globals():
    raise RuntimeError("Run the existing final lesson-topic cleaning cells first.")
FINAL_COLUMNS = [
    "c_id", "c_h", "c_r_", "lesson_topic", "lesson_topic_clean", "e", "e_clean",
    "z_what", "z_what_clean", "z_why", "original_split", "source_row_index",
]
REMOVED_COLUMNS = ["e_original", "z_what_original", "c_r", "c_revision"]
EXPECTED_FINAL_SPLIT_COUNTS = {"train": 397, "validation": 68, "test": 195}
simplification_source_dataset = final_dataset
simplification_source_snapshot = {
    split: deepcopy(list(records)) for split, records in final_dataset.items()
}
excluded_history_audit_path = Path("data") / "bridge_excluded.jsonl"
excluded_history_audit_before = excluded_history_audit_path.read_bytes()

schema_issue_rows = []
redundancy_mismatch_rows = []
expected_source_columns = set(FINAL_COLUMNS) | set(REMOVED_COLUMNS)
expected_message_fields = {"id", "text", "time", "user"}
expected_message_roles = {"tutor", "student"}
source_turn_count = 0
source_role_counts = Counter()

def record_schema_issue(split, row, issue, turn_index=None, turn_id=None):
    schema_issue_rows.append({
        "c_id": row.get("c_id"), "original_split": split,
        "source_row_index": row.get("source_row_index"),
        "turn_index": turn_index, "turn_id": turn_id, "issue": issue,
    })

for split, rows in simplification_source_snapshot.items():
    for row in rows:
        missing = expected_source_columns - set(row)
        unexpected = set(row) - expected_source_columns
        if missing or unexpected:
            record_schema_issue(split, row, f"Record fields: missing={sorted(missing)}, unexpected={sorted(unexpected)}")
        for original_copy, original_field in (("e_original", "e"), ("z_what_original", "z_what")):
            if row.get(original_copy) != row.get(original_field):
                redundancy_mismatch_rows.append({
                    "c_id": row.get("c_id"), "original_split": split,
                    "source_row_index": row.get("source_row_index"),
                    "copy_field": original_copy, "original_field": original_field,
                    "copy_value": row.get(original_copy), "original_value": row.get(original_field),
                })
        history = row.get("c_h")
        if not isinstance(history, list):
            record_schema_issue(split, row, "c_h is missing or is not a list")
            continue
        source_turn_count += len(history)
        for turn_index, message in enumerate(history):
            if not isinstance(message, dict):
                record_schema_issue(split, row, "Message is not a dictionary", turn_index)
                continue
            missing = expected_message_fields - set(message)
            unexpected = set(message) - expected_message_fields
            if missing or unexpected:
                record_schema_issue(split, row, f"Message fields: missing={sorted(missing)}, unexpected={sorted(unexpected)}",
                                    turn_index, message.get("id"))
            role = message.get("user")
            if role not in expected_message_roles:
                record_schema_issue(split, row, f"Unexpected speaker label: {role!r}", turn_index, message.get("id"))
            else:
                source_role_counts[role] += 1
            if not isinstance(message.get("text"), str):
                record_schema_issue(split, row, "Message text is not a string", turn_index, message.get("id"))
            if not isinstance(message.get("id"), int) or isinstance(message.get("id"), bool):
                record_schema_issue(split, row, "Unexpected turn identifier type", turn_index, message.get("id"))

history_schema_issues = pd.DataFrame(schema_issue_rows, columns=[
    "c_id", "original_split", "source_row_index", "turn_index", "turn_id", "issue",
])
annotation_copy_mismatches = pd.DataFrame(redundancy_mismatch_rows)
if not history_schema_issues.empty:
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(history_schema_issues)
else:
    print("No missing/unexpected record or message fields, invalid message structures, or unexpected speakers.")
if not annotation_copy_mismatches.empty:
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(annotation_copy_mismatches)
else:
    print("Verified all 660 records: e_original == e and z_what_original == z_what.")
assert history_schema_issues.empty, "Inspect affected c_id values above; export is blocked."
assert annotation_copy_mismatches.empty, "Annotation copies differ; do not remove them."
assert {split: len(rows) for split, rows in simplification_source_snapshot.items()} == EXPECTED_FINAL_SPLIT_COUNTS
print("Conversation turns to reformat:", source_turn_count)
print("Speaker counts:", dict(source_role_counts))

No missing/unexpected record or message fields, invalid message structures, or unexpected speakers.
Verified all 660 records: e_original == e and z_what_original == z_what.
Conversation turns to reformat: 2640
Speaker counts: {'tutor': 1593, 'student': 1047}


### 6.2 Reformat Histories and Select the 12 Final Fields

Rename `id` to `turn_id` and `user` to `role`; keep the key `text` and omit only `time`. Cleaned messages contain exactly `turn_id`, `role`, and `text`. Preserve all text exactly, including whitespace and mathematical mistakes. Preserve original IDs, message order, short replies, and consecutive turns from the same speaker.

`history_annotated_dataset` retains original `c_h` alongside `c_h_clean`. `simplified_dataset` contains exactly the 12 requested fields in their specified order; its `c_h` stores the reformatted messages, without a separate `c_h_clean` field. Expert responses and labels are copied unchanged.


In [2]:
def reformat_history(record):
    return {"c_h_clean": [
        {"turn_id": message["id"], "role": message["user"], "text": message["text"]}
        for message in record["c_h"]
    ]}

cleaned_history_feature = List({
    "turn_id": Value("int64"), "role": Value("string"), "text": Value("string"),
})
history_annotated_dataset = DatasetDict({
    split: records.map(
        reformat_history,
        features=Features({**records.features, "c_h_clean": cleaned_history_feature}),
        keep_in_memory=True, load_from_cache_file=False,
        desc=f"Reformat histories: {split}",
    )
    for split, records in simplification_source_dataset.items()
})
simplified_dataset = DatasetDict()
for split, records in history_annotated_dataset.items():
    output_features = Features({
        column: cleaned_history_feature if column == "c_h" else records.features[column]
        for column in FINAL_COLUMNS
    })
    simplified_dataset[split] = Dataset.from_list([
        {column: record["c_h_clean"] if column == "c_h" else record[column]
         for column in FINAL_COLUMNS}
        for record in records
    ], features=output_features)

print("Example c_h before reformatting:")
example_history_before = simplification_source_snapshot["train"][0]["c_h"]
example_history_after = simplified_dataset["train"][0]["c_h"]
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(pd.DataFrame(example_history_before))
    print("Same c_h after reformatting:")
    display(pd.DataFrame(example_history_after))
print("Removed columns:", REMOVED_COLUMNS)
print("Final columns:", FINAL_COLUMNS)

Example c_h before reformatting:


,id,text,time,user
0,19,Sounds great.,16:19,tutor
1,20,Use the area formula to find the area of the current patio.,16:25,tutor
2,21,What is the area of the pink rectangle?,16:37,tutor
3,22,4 m,17:13,student


Same c_h after reformatting:


,turn_id,role,text
0,19,tutor,Sounds great.
1,20,tutor,Use the area formula to find the area of the current patio.
2,21,tutor,What is the area of the pink rectangle?
3,22,student,4 m


Removed columns: ['e_original', 'z_what_original', 'c_r', 'c_revision']
Final columns: ['c_id', 'c_h', 'c_r_', 'lesson_topic', 'lesson_topic_clean', 'e', 'e_clean', 'z_what', 'z_what_clean', 'z_why', 'original_split', 'source_row_index']


### 6.3 Validate Records, Histories, Labels, and Schema

Compare every reformatted history turn-by-turn with the original. Check exact IDs, roles, text, message counts, and order; verify that every other retained field is unchanged. Use `(original_split, source_row_index)` for record identity because case IDs alone can recur.

Validation failures are displayed with affected case IDs and block export. All original source records remain unchanged in memory and are preserved on disk in the audit snapshot before final exports are replaced.


In [3]:
history_schema_validation_passed = False
history_validation_issue_rows = []
seen_simplified_keys = set()
simplified_record_count = 0
simplified_turn_count = 0

def add_history_validation_issue(split, row, issue, turn_index=None):
    history_validation_issue_rows.append({
        "c_id": row.get("c_id"), "original_split": split,
        "source_row_index": row.get("source_row_index"),
        "turn_index": turn_index, "issue": issue,
    })

if {split: len(records) for split, records in simplified_dataset.items()} != EXPECTED_FINAL_SPLIT_COUNTS:
    history_validation_issue_rows.append({"issue": "Final split record counts differ from 397/68/195."})
for split, records in simplified_dataset.items():
    originals = simplification_source_snapshot[split]
    if list(simplification_source_dataset[split]) != originals:
        history_validation_issue_rows.append({"original_split": split, "issue": "Source records changed."})
    for index, (original, record) in enumerate(zip(originals, records)):
        simplified_record_count += 1
        if list(record) != FINAL_COLUMNS:
            add_history_validation_issue(split, record, "Final fields or their order differ from FINAL_COLUMNS.")
        if record["original_split"] != split:
            add_history_validation_issue(split, record, "Original split changed.")
        key = (record["original_split"], record["source_row_index"])
        if key in seen_simplified_keys:
            add_history_validation_issue(split, record, "Source record duplicated.")
        seen_simplified_keys.add(key)
        for column in FINAL_COLUMNS:
            if column != "c_h" and record[column] != original[column]:
                add_history_validation_issue(split, record, f"Retained field changed: {column}")
        history = record["c_h"]
        if not isinstance(history, list):
            add_history_validation_issue(split, record, "Reformatted c_h is not a list.")
            continue
        simplified_turn_count += len(history)
        if len(history) != len(original["c_h"]):
            add_history_validation_issue(split, record, "Number of messages changed.")
        for turn_index, (before, after) in enumerate(zip(original["c_h"], history)):
            expected = {"turn_id": before["id"], "role": before["user"], "text": before["text"]}
            if set(after) != {"turn_id", "role", "text"}:
                add_history_validation_issue(split, record, "Unexpected/missing cleaned message keys.", turn_index)
            if after != expected:
                add_history_validation_issue(split, record, "Turn ID, speaker, content, or message order changed.", turn_index)
        if history_annotated_dataset[split][index]["c_h"] != original["c_h"]:
            add_history_validation_issue(split, record, "Intermediate original c_h changed.")
        if history_annotated_dataset[split][index]["c_h_clean"] != history:
            add_history_validation_issue(split, record, "Intermediate c_h_clean differs from final c_h.")

history_validation_issues = pd.DataFrame(history_validation_issue_rows)
if not history_validation_issues.empty:
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(history_validation_issues)
assert history_validation_issues.empty, "Inspect affected c_id values above; do not export."
assert simplified_record_count == len(seen_simplified_keys) == 660
assert simplified_turn_count == source_turn_count
assert all(records.column_names == FINAL_COLUMNS for records in simplified_dataset.values())
assert all(len(record) == 12 for records in simplified_dataset.values() for record in records)
assert all(record["e_original"] == record["e"] and record["z_what_original"] == record["z_what"]
           for records in simplification_source_snapshot.values() for record in records)
history_schema_validation_passed = True
simplified_split_counts = pd.DataFrame([
    {"split": split, "records": len(records), "turns": sum(len(row["c_h"]) for row in records)}
    for split, records in simplified_dataset.items()
]).set_index("split")
print(f"Validation passed: {simplified_record_count} records and {simplified_turn_count} turns reformatted.")
print("All retained labels, expert responses, other retained fields, and memberships are unchanged.")
display(simplified_split_counts)

Validation passed: 660 records and 2640 turns reformatted.
All retained labels, expert responses, other retained fields, and memberships are unchanged.


,records,turns
split,,
train,397,1588
validation,68,272
test,195,780


### 6.4 Preserve Audit Snapshots and Write Final Exports

First save the complete pre-format 16-field records under `data/audit/before_history_reformat/`. Existing audit snapshots are verified rather than overwritten. They preserve original histories with timestamps, both redundant annotation copies, `c_r`, and `c_revision`.

Then update only the three shared split files under `data/`. Reload each JSONL file and verify record counts, exact field order, and the cleaned conversation structure. The existing 40-record `bridge_excluded.jsonl` is left unchanged. `simplified_dataset` is the final 12-field in-memory version; the earlier `final_dataset` remains available for audit and reproducibility.


In [4]:
assert history_schema_validation_passed, "Complete all history/schema checks before exporting."
history_audit_directory = Path("data") / "audit" / "before_history_reformat"
history_output_directory = Path("data")
history_audit_directory.mkdir(parents=True, exist_ok=True)

# Check all existing audit snapshots before changing any export.
for split, original_rows in simplification_source_snapshot.items():
    audit_path = history_audit_directory / f"bridge_{split}.jsonl"
    if audit_path.exists():
        with audit_path.open(encoding="utf-8") as handle:
            saved_audit = [json.loads(line) for line in handle]
        assert saved_audit == original_rows, f"Audit snapshot differs: {audit_path}; preserve and review it."

def write_history_jsonl(path, records):
    with path.open("w", encoding="utf-8", newline=chr(10)) as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + chr(10))

def read_history_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]

for split, original_rows in simplification_source_snapshot.items():
    audit_path = history_audit_directory / f"bridge_{split}.jsonl"
    if not audit_path.exists():
        write_history_jsonl(audit_path, original_rows)
    assert read_history_jsonl(audit_path) == original_rows

simplified_export_manifest = []
for split, records in simplified_dataset.items():
    output_path = history_output_directory / f"bridge_{split}.jsonl"
    write_history_jsonl(output_path, records)
    reloaded = read_history_jsonl(output_path)
    assert reloaded == list(records)
    assert len(reloaded) == EXPECTED_FINAL_SPLIT_COUNTS[split]
    assert all(list(row) == FINAL_COLUMNS for row in reloaded)
    for original, record in zip(simplification_source_snapshot[split], reloaded):
        assert len(record["c_h"]) == len(original["c_h"])
        for before, after in zip(original["c_h"], record["c_h"]):
            assert list(after) == ["turn_id", "role", "text"]
            assert after == {"turn_id": before["id"], "role": before["user"], "text": before["text"]}
    simplified_export_manifest.append({"file": output_path.as_posix(), "records": len(reloaded)})
assert excluded_history_audit_path.read_bytes() == excluded_history_audit_before
assert len(read_history_jsonl(excluded_history_audit_path)) == 40
print("Exported and reloaded all three 12-field JSONL files:")
display(pd.DataFrame(simplified_export_manifest))
print("Full pre-format audit snapshots:", history_audit_directory.as_posix())
print("Excluded audit preserved: 40 records.")

Exported and reloaded all three 12-field JSONL files:


,file,records
0,data/bridge_train.jsonl,397
1,data/bridge_validation.jsonl,68
2,data/bridge_test.jsonl,195


Full pre-format audit snapshots: data/audit/before_history_reformat
Excluded audit preserved: 40 records.


### 6.5 Separate Agent Inputs from Evaluation References

The Student Analyser may receive `c_h` and `lesson_topic_clean`. Keep expert annotations `e`, `e_clean`, `z_what`, `z_what_clean`, `z_why`, and `c_r_` hidden from agent inference and accessible only to the evaluation pipeline. The dataset intentionally retains these reference fields for evaluation; their presence in a record does not authorise passing the whole record to an agent.

No inference pipeline or agent implementation is created by these preprocessing cells.


<a id="segment-7"></a>

## Segment 7: Conversation Context Analysis

### 7.1 Define the Analysis Scope

This read-only analysis uses `simplified_dataset`, the final 660-record dataset with reformatted `c_h`. It creates separate DataFrames and does not add fields to the dataset, change records, or write JSONL files.

We distinguish **a visible mathematical task** from **enough information to evaluate the latest student response**. A question about a pink rectangle may omit the diagram or dimensions; an explicit calculation such as `36+42` supplies its operands. A generic follow-up such as "Is that your final answer?" does not by itself reveal the mathematics problem. Short replies like "yes" are retained and flagged for interpretation, because they may be legitimate responses to yes/no questions.

**Method:** deterministic, conservative screening based only on `c_h`, followed by reviewable examples. Counts describe rule-detected candidates, not verified ground truth. Rules can miss paraphrases, and visual/referential language does not prove that information is missing. Cases not detected as self-contained must not automatically be called unevaluable.

The analysis never uses `e`, `e_clean`, `z_what`, `z_what_clean`, `z_why`, `c_r_`, or future turns to reconstruct a question. Manual-review fields remain unfilled until a person reviews each case. No student answers are automatically graded.


In [5]:
import re
from collections import Counter
from copy import deepcopy
import pandas as pd
from IPython.display import display, Markdown

if "simplified_dataset" not in globals():
    raise RuntimeError("Run the final history/schema section before this analysis.")
context_audit_source_reference = simplified_dataset
context_audit_source_snapshot = {
    split: deepcopy(list(records)) for split, records in simplified_dataset.items()
}
assert {split: len(records) for split, records in simplified_dataset.items()} == {
    "train": 397, "validation": 68, "test": 195,
}

# These patterns are screening signals, not correctness judgments.
MATH_WORD_PATTERN = re.compile(
    r"\b(?:add(?:ing|ition)?|subtract(?:ing|ion)?|sum|difference|product|quotient|"
    r"multiply(?:ing)?|multiplying|multiplication|divide|dividing|division|times|minus|plus|"
    r"number|numbers|digit|digits|value|place|tens|ones|hundred|thousand|million|"
    r"fraction|fractions|decimal|decimals|percent|percentage|ratio|rate|unit|factor|multiple|"
    r"prime|composite|even|odd|round|rounding|equation|expression|solve|simplify|"
    r"area|perimeter|volume|length|width|height|angle|shape|shapes|figure|figures|"
    r"rectangle|rectangles|triangle|quadrilateral|polygon|symmetry|square|circle|"
    r"axis|axes|coordinate|coordinates|slope|graph|plot|grid|array|numerator|denominator|"
    r"count|counting|measure|measurement|convert|conversion|estimate|estimation|"
    r"probability|mean|median|range|pi|hour|hours|minute|minutes|second|seconds|"
    r"pound|pounds|ounce|ounces|feet|foot|inch|inches|meter|meters|litre|liter)\b",
    re.IGNORECASE,
)
TASK_CUE_PATTERN = re.compile(
    r"\b(?:what|how|which|where|why|is|are|does|do|can|could|"
    r"find|calculate|compute|solve|evaluate|convert|round|write|name|identify|compare|"
    r"simplify|determine|count|tell|add|subtract|multiply|divide|estimate|give)\b|[?]",
    re.IGNORECASE,
)
GENERIC_PROMPT_PATTERN = re.compile(
    r"\b(?:question|problem|answer|final answer|give it a try|give it a shot|"
    r"go ahead|do you understand|are you sure|your work|try again|read carefully)\b",
    re.IGNORECASE,
)
EXTERNAL_REFERENCE_PATTERNS = {
    "visual_material": re.compile(
        r"\b(?:whiteboard|white board|picture|image|diagram|drawing|graph|grid|array|"
        r"chart|table|dot|plotted|shown|shaded|unshaded|pink|blue|red|green|yellow)\b", re.I
    ),
    "referential_object": re.compile(
        r"\b(?:this|that|the given|the above|the following|top|bottom|left|right)\s+"
        r"(?:question|problem|figure|shape|rectangle|triangle|number|point|graph|expression|equation)|"
        r"\b(?:above|below|on the board|in the picture|in the figure|option [a-d]|choices)\b", re.I
    ),
    "unspecified_geometry": re.compile(
        r"\b(?:area|perimeter|volume|length|width|height)\b.*"
        r"\b(?:the|this|that|our)\s+(?:pink\s+|top\s+|bottom\s+)?"
        r"(?:rectangle|triangle|figure|shape|garden|wood|patio|prism)\b", re.I
    ),
}
NUMBER_TOKEN_PATTERN = re.compile(r"(?<![A-Za-z0-9])[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?![A-Za-z0-9])")
NUMERIC_OPERATION_PATTERN = re.compile(
    r"\d(?:[\d,.]*\d)?\s*(?:[+*/^-]|\u00d7|\u00f7|x|plus|minus|times|"
    r"multiplied by|divided by)\s*[-+]?\d", re.I
)
WHOLE_EXPRESSION_PATTERN = re.compile(
    r"^\s*[-+]?\d[\d,.]*\s*(?:[+*/^-]|\u00d7|\u00f7|x)\s*[-+]?\d[\d,.]*\s*[?]?\s*$", re.I
)
BRIEF_REPLY_PATTERN = re.compile(
    r"^(?:yes|yeah|yep|no|nope|ok|okay|alright|right|sure|done|next(?: lesson)?|"
    r"thank you|thanks)(?:\s+(?:mam|maam|sir|miss|teacher))?[.!?,]*$", re.I
)
UNCERTAINTY_PATTERN = re.compile(
    r"\b(?:idk|i don.t know|i dont know|no clue|not sure|confused|don.t understand|help me)\b", re.I
)
ANSWER_WORD_PATTERN = re.compile(
    r"\b(?:rectangle|triangle|square|rhombus|parallelogram|trapezoid|quadrilateral|"
    r"circle|polygon|pentagon|hexagon|octagon|cube|prism|cylinder|cone|sphere|"
    r"area|perimeter|volume|add|adding|subtract|subtracting|multiply|multiplying|"
    r"divide|dividing|fraction|decimal|numerator|denominator|positive|negative|"
    r"greater|less|equal|horizontal|vertical|parallel|perpendicular|tens|ones|"
    r"hundreds|thousands)\b", re.I
)

### 7.2 Screen Question Visibility and Response Context

Inspect the history only up to the latest student message. Keep the most recent mathematical task candidate and its exact text as evidence. Strong self-contained signals are limited to explicit numerical operations, complete named-operand questions, specified rounding/place-value tasks, and a few fully stated factual or definition questions.

A task is marked **potentially self-contained**, not confirmed sufficient. The latest-response screen separately flags brief confirmations, uncertainty/help requests, and responses containing numbers or mathematical terms. A candidate for answer evaluation also needs the detected task to be the immediately preceding tutor prompt. Other records remain review candidates.


In [6]:
def prompt_screen_signals(text):
    explicit_expression = bool(WHOLE_EXPRESSION_PATTERN.fullmatch(text))
    mathematical = bool(MATH_WORD_PATTERN.search(text) or NUMERIC_OPERATION_PATTERN.search(text))
    task = explicit_expression or (mathematical and bool(TASK_CUE_PATTERN.search(text)))
    return task, mathematical


def self_contained_task_signals(text):
    reasons = []
    numbers = NUMBER_TOKEN_PATTERN.findall(text)
    low = " ".join(text.casefold().split())
    if NUMERIC_OPERATION_PATTERN.search(text):
        reasons.append("numeric operation with explicit operands")
    if len(numbers) >= 2 and re.search(r"\b(?:sum|product|difference|quotient) of\b", low):
        reasons.append("named operation with two stated numbers")
    if len(numbers) >= 2 and re.search(r"how many times.*(?:go|fit|into)", low):
        reasons.append("division question with two stated numbers")
    if numbers and re.search(r"\bround", low) and re.search(
        r"nearest|decimal place|significant|tenth|hundredth|thousandth", low
    ):
        reasons.append("rounding question includes a number and precision")
    if numbers and re.search(r"place value|value of.*(?:in the number|in \d)|"
                             r"(?:hundreds|thousands|tens|ones) place", low):
        reasons.append("place-value task with a stated number")
    if len(numbers) >= 2 and re.search(r"compare|greater|less|larger|smaller", low):
        reasons.append("comparison with stated numbers")
    property_question = re.search(
        r"\b(?:is|are)\b.*\b(?:prime|composite|even|odd)\b", low
    )
    factor_question = re.search(
        r"\b(?:find|what are|list|identify|give|tell)\b.*\bfactors?\b.*\b(?:of|for)\b.*\d", low
    )
    if numbers and (property_question or factor_question):
        reasons.append("explicit number-property question with a stated number")
    if re.search(r"write.*(?:whole number|standard form|number).*(?:hundred|thousand|million)", low):
        reasons.append("number-word writing task visible in text")
    if re.search(r"what (?:is|does).*\b(?:addition|subtraction|multiplication|division|perimeter|area|volume|fraction|denominator|numerator)\b", low):
        if not re.search(r"\b(?:this|that|the|our|given|top|bottom|pink)\b", low):
            reasons.append("standalone definition question candidate")
    if re.search(r"(?:what.*value of pi|how many hours.*(?:a|one) day)", low):
        reasons.append("fully stated factual question candidate")
    return reasons


def screen_latest_student_reply(text):
    low = " ".join(text.casefold().split())
    if not low:
        return "empty_reply"
    if BRIEF_REPLY_PATTERN.fullmatch(low):
        return "brief_confirmation_or_acknowledgement"
    if UNCERTAINTY_PATTERN.search(low):
        return "uncertainty_or_help_request"
    if NUMBER_TOKEN_PATTERN.search(low) or ANSWER_WORD_PATTERN.search(low):
        return "number_or_math_term_present"
    if "?" in low or re.search(r"\b(?:what|how|why|is it right|is this right)\b", low):
        return "clarification_or_question"
    return "other_text_needs_interpretation"


def screen_conversation_context(history):
    student_indices = [index for index, message in enumerate(history) if message["role"] == "student"]
    if not student_indices:
        return {
            "question_screen": "no_student_turn", "context_screen": "no_student_turn",
            "evaluation_screen": "no_student_turn", "latest_student_response": None,
            "latest_student_turn_id": None, "reply_screen": "no_student_turn",
            "task_turn_id": None, "task_text": None, "task_role": None,
            "self_contained_signals": [], "reference_signals": [],
            "immediate_tutor_task": False, "latest_tutor_text": None,
        }
    latest_index = student_indices[-1]
    latest = history[latest_index]
    available = history[:latest_index]
    tasks = [(index, message) for index, message in enumerate(available)
             if prompt_screen_signals(message["text"])[0]]
    tutor_tasks = [(index, message) for index, message in tasks if message["role"] == "tutor"]
    selected = tutor_tasks[-1] if tutor_tasks else (tasks[-1] if tasks else None)
    tutor_indices = [index for index, message in enumerate(available) if message["role"] == "tutor"]
    latest_tutor_index = tutor_indices[-1] if tutor_indices else None
    latest_tutor_text = available[latest_tutor_index]["text"] if latest_tutor_index is not None else None
    reply_screen = screen_latest_student_reply(latest["text"])
    task_text = selected[1]["text"] if selected else None
    immediate = bool(selected and selected[1]["role"] == "tutor" and selected[0] == latest_index - 1)
    strong_signals = self_contained_task_signals(task_text) if task_text else []
    reference_text = task_text if task_text else " ".join(message["text"] for message in available)
    references = [name for name, pattern in EXTERNAL_REFERENCE_PATTERNS.items() if pattern.search(reference_text)]
    if selected:
        question_screen = "explicit_math_task_candidate"
    elif any(GENERIC_PROMPT_PATTERN.search(message["text"]) for message in available):
        question_screen = "referential_or_generic_prompt_only"
    elif any(prompt_screen_signals(message["text"])[1] for message in available):
        question_screen = "math_content_without_detected_task"
    else:
        question_screen = "no_task_signal_detected"
    if strong_signals:
        context_screen = "potentially_self_contained_task"
    elif selected and references:
        context_screen = "referential_or_visual_task_needs_review"
    elif selected:
        context_screen = "task_found_context_needs_review"
    else:
        context_screen = "no_math_task_identified"
    if strong_signals and immediate and reply_screen == "number_or_math_term_present":
        evaluation_screen = "candidate_for_answer_evaluation"
    elif strong_signals and not immediate:
        evaluation_screen = "task_reply_alignment_needs_review"
    elif strong_signals:
        evaluation_screen = "response_meaning_needs_review"
    elif selected and references:
        evaluation_screen = "referential_context_needs_review"
    elif selected:
        evaluation_screen = "task_context_needs_review"
    else:
        evaluation_screen = "no_identified_task_for_latest_reply"
    return {
        "question_screen": question_screen, "context_screen": context_screen,
        "evaluation_screen": evaluation_screen,
        "latest_student_response": latest["text"],
        "latest_student_turn_id": latest["turn_id"], "reply_screen": reply_screen,
        "task_turn_id": selected[1]["turn_id"] if selected else None,
        "task_text": task_text, "task_role": selected[1]["role"] if selected else None,
        "self_contained_signals": strong_signals, "reference_signals": references,
        "immediate_tutor_task": immediate, "latest_tutor_text": latest_tutor_text,
    }

context_audit = pd.DataFrame([
    {
        "c_id": record["c_id"], "original_split": split,
        "source_row_index": record["source_row_index"],
        "history_turn_count": len(record["c_h"]),
        "c_h": deepcopy(record["c_h"]),
        **screen_conversation_context(record["c_h"]),
    }
    for split, records in simplified_dataset.items() for record in records
])
assert len(context_audit) == 660

### 7.3 Summarise Screening Results by Split

Use the tables below as screening estimates. A detected task can still depend on an omitted diagram; an undetected task may be phrased outside the rules. Only a manual c_h-only audit can establish the final numbers of complete questions and sufficient contexts. The candidate answer-evaluation count is neither a proven lower bound nor an upper bound.


In [7]:
def context_screen_counts(column):
    counts = context_audit[column].value_counts().rename("records").to_frame()
    counts["percent"] = (100 * counts["records"] / len(context_audit)).round(1)
    return counts

context_question_counts = context_screen_counts("question_screen")
context_sufficiency_counts = context_screen_counts("context_screen")
context_evaluation_counts = context_screen_counts("evaluation_screen")
context_reply_counts = context_screen_counts("reply_screen")
print("Window lengths:")
display(context_audit["history_turn_count"].value_counts().sort_index().rename("records").to_frame())
print("Question/task visibility screen:")
display(context_question_counts)
print("Task completeness screen (not verified sufficiency):")
display(context_sufficiency_counts)
print("Latest-response evaluation screen:")
display(context_evaluation_counts)
print("Latest student response type screen:")
display(context_reply_counts)

print("Question screen by original split:")
context_question_by_split = pd.crosstab(
    context_audit["question_screen"], context_audit["original_split"]
).reindex(columns=["train", "validation", "test"], fill_value=0)
display(context_question_by_split)
print("Evaluation screen by original split:")
context_evaluation_by_split = pd.crosstab(
    context_audit["evaluation_screen"], context_audit["original_split"]
).reindex(columns=["train", "validation", "test"], fill_value=0)
display(context_evaluation_by_split)

print("Brief replies are not automatically considered incorrect or unusable:")
display(pd.crosstab(context_audit["reply_screen"], context_audit["context_screen"]))
print("First 20 per-record screening results (full history remains in context_audit['c_h']):")
with pd.option_context("display.max_colwidth", 120):
    display(context_audit[[
        "c_id", "original_split", "source_row_index", "task_text", "latest_student_response",
        "question_screen", "context_screen", "evaluation_screen",
    ]].head(20))

Window lengths:


,records
history_turn_count,
4,660


Question/task visibility screen:


,records,percent
question_screen,,
explicit_math_task_candidate,377,57.1
referential_or_generic_prompt_only,242,36.7
no_task_signal_detected,28,4.2
math_content_without_detected_task,13,2.0


Task completeness screen (not verified sufficiency):


...

original_split,train,validation,test
evaluation_screen,,,
candidate_for_answer_evaluation,43,10,29
no_identified_task_for_latest_reply,169,25,89
referential_context_needs_review,20,11,11
response_meaning_needs_review,19,2,6
task_context_needs_review,119,16,55
task_reply_alignment_needs_review,27,4,5


Brief replies are not automatically considered incorrect or unusable:


context_screen,no_math_task_identified,potentially_self_contained_task,referential_or_visual_task_needs_review,task_found_context_needs_review
reply_screen,,,,
brief_confirmation_or_acknowledgement,109,14,3,29
clarification_or_question,3,3,2,2
empty_reply,0,2,0,0
number_or_math_term_present,106,97,25,118
other_text_needs_interpretation,64,27,12,41
uncertainty_or_help_request,1,2,0,0


First 20 per-record screening results (full history remains in context_audit['c_h']):


,c_id,original_split,source_row_index,task_text,latest_student_response,question_screen,context_screen,evaluation_screen
0,4110653_19,train,0,What is the area of the pink rectangle?,4 m,explicit_math_task_candidate,referential_or_visual_task_needs_review,referential_context_needs_review
1,2950057_118,train,1,NaN,0.8,referential_or_generic_prompt_only,no_math_task_identified,no_identified_task_for_latest_reply
2,2989485_22,train,2,NaN,yes,referential_or_generic_prompt_only,no_math_task_identified,no_identified_task_for_latest_reply
3,2928616_65,train,3,What is the value of 36+42?,"1,512",explicit_math_task_candidate,potentially_self_contained_task,candidate_for_answer_evaluation
4,4054798_68,train,4,"Since 16.50 and 12.95 have less than 2 tens, we know that they are both less than 20.",a little bit,explicit_math_task_candidate,potentially_self_contained_task,task_reply_alignment_needs_review
5,4055603_103,train,5,NaN,nhw,referential_or_generic_prompt_only,no_math_task_identified,no_identified_task_for_latest_reply
6,2916421_82,train,6,NaN,no mam,referential_or_generic_prompt_only,no_math_task_identified,no_identified_task_for_latest_reply
7,2963055_17,train,7,NaN,2 inches,no_task_signal_detected,no_math_task_identified,no_identified_task_for_latest_reply
8,6172033_30,train,8,How many times 27 can go into the 4th fact?,4 times in,explicit_math_task_candidate,task_found_context_needs_review,task_context_needs_review
9,6165953_133,train,9,What is the name of quadrilateral having the length bigger than width?,rhombus,explicit_math_task_candidate,task_found_context_needs_review,task_context_needs_review


### 7.4 Inspect Histories and Prepare Manual Review

The examples below show up to two histories per evaluation-screen category. Use `show_context_audit_case("train", source_row_index)` to inspect another record by its stable source identity.

`context_manual_review` contains all 660 records and empty fields for manual question visibility (`yes` / `no` / `uncertain`), context sufficiency for the latest response (`yes` / `no` / `uncertain` / `not_applicable`), and review notes. Decide sufficiency using only the history, not reference labels or expert responses. A mathematically complete question can still require interpretation of the student's response. Review all categories, including the automatically positive candidates, before treating the counts as confirmed.


In [8]:
def show_context_audit_case(split, source_row_index):
    matches = context_audit.loc[
        context_audit["original_split"].eq(split)
        & context_audit["source_row_index"].eq(source_row_index)
    ]
    if len(matches) != 1:
        raise ValueError("Expected exactly one case for the supplied split and source row.")
    case = matches.iloc[0]
    display(Markdown(f"##### {case['c_id']} ({split}, source row {source_row_index})"))
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(pd.DataFrame([case[[
            "question_screen", "context_screen", "evaluation_screen", "reply_screen",
            "task_text", "latest_student_response", "self_contained_signals", "reference_signals",
        ]].to_dict()]))
        display(pd.DataFrame(case["c_h"]))

for category in context_audit["evaluation_screen"].drop_duplicates():
    display(Markdown(f"#### Screening examples: {category}"))
    examples = context_audit.loc[context_audit["evaluation_screen"].eq(category)].head(2)
    for _, example in examples.iterrows():
        show_context_audit_case(example["original_split"], example["source_row_index"])

context_manual_review = context_audit.copy(deep=True)
context_manual_review["manual_question_present"] = None
context_manual_review["manual_context_sufficient"] = None
context_manual_review["manual_review_notes"] = None
print("Manual-review table prepared:", len(context_manual_review), "cases; none manually adjudicated yet.")

#### Screening examples: referential_context_needs_review

##### 4110653_19 (train, source row 0)

,question_screen,context_screen,evaluation_screen,reply_screen,task_text,latest_student_response,self_contained_signals,reference_signals
0,explicit_math_task_candidate,referential_or_visual_task_needs_review,referential_context_needs_review,number_or_math_term_present,What is the area of the pink rectangle?,4 m,[],"[visual_material, unspecified_geometry]"


,turn_id,role,text
0,19,tutor,Sounds great.
1,20,tutor,Use the area formula to find the area of the current patio.
2,21,tutor,What is the area of the pink rectangle?
3,22,student,4 m


##### 4110653_56 (train, source row 10)

...

,turn_id,role,text
0,49,tutor,There are 0 tens in the number.
1,50,tutor,There are 9 ones in the number.
2,51,tutor,How would you write the whole number four hundred nine?
3,52,student,is it right


##### 4289415_22 (train, source row 46)

,question_screen,context_screen,evaluation_screen,reply_screen,task_text,latest_student_response,self_contained_signals,reference_signals
0,explicit_math_task_candidate,potentially_self_contained_task,response_meaning_needs_review,other_text_needs_interpretation,what is addition?,-,[standalone definition question candidate],[]


,turn_id,role,text
0,22,student,now i can
1,23,tutor,Good!
2,24,tutor,what is addition?
3,25,student,-


Manual-review table prepared: 660 cases; none manually adjudicated yet.


### 7.5 Validate the Analysis Boundaries

These checks verify that every original record appears once, all split memberships and histories are unchanged, and screening decisions do not use expert annotations. No dataset rows or fields are added, removed, or rewritten. The heuristic examples are basic consistency checks; they do not establish measured classification accuracy.


In [9]:
assert simplified_dataset is context_audit_source_reference
for split, records in simplified_dataset.items():
    assert list(records) == context_audit_source_snapshot[split]
assert not context_audit.duplicated(["original_split", "source_row_index"]).any()
assert context_audit.groupby("original_split").size().to_dict() == {"train": 397, "validation": 68, "test": 195}
assert len(context_manual_review) == len(context_audit) == 660
for table in (context_question_counts, context_sufficiency_counts, context_evaluation_counts, context_reply_counts):
    assert int(table["records"].sum()) == 660
assert not {"e", "e_clean", "z_what", "z_what_clean", "z_why", "c_r_"}.intersection(context_audit.columns)
source_histories_by_key = {
    (split, row["source_row_index"]): row["c_h"]
    for split, rows in context_audit_source_snapshot.items() for row in rows
}
for _, row in context_audit.iterrows():
    assert row["c_h"] == source_histories_by_key[(row["original_split"], row["source_row_index"])]
    assert screen_conversation_context(row["c_h"])["evaluation_screen"] == row["evaluation_screen"]
# Direct arithmetic versus visual and generic prompts: transparent rule examples.
example_cases = [
    ([{"turn_id": 1, "role": "tutor", "text": "What is the value of 36+42?"},
      {"turn_id": 2, "role": "student", "text": "1512"}], "candidate_for_answer_evaluation"),
    ([{"turn_id": 1, "role": "tutor", "text": "What is the area of the pink rectangle?"},
      {"turn_id": 2, "role": "student", "text": "4 m"}], "referential_context_needs_review"),
    ([{"turn_id": 1, "role": "tutor", "text": "Is that your final answer?"},
      {"turn_id": 2, "role": "student", "text": "0.8"}], "no_identified_task_for_latest_reply"),
]
for history, expected in example_cases:
    assert screen_conversation_context(history)["evaluation_screen"] == expected
print("Analysis validated: all 660 records and original histories preserved.")
print("No dataset or JSONL export was changed. Screening counts require manual confirmation.")

Analysis validated: all 660 records and original histories preserved.
No dataset or JSONL export was changed. Screening counts require manual confirmation.
